In [ ]:

# Parent directory containing class folders
parent_dir = 'D:/arpon/ARPON'

def find_and_fix_images(directory):
    """
    Check for corrupted images and fix images with an alpha channel.

    Args:
        directory (str): Path to the directory to check.

    Returns:
        int: Number of corrupted files removed.
        int: Number of images fixed (alpha channel removed).
    """
    corrupted_count = 0
    fixed_count = 0
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if file_path.lower().endswith(('.jpeg', '.jpg', '.jpeg', '.bmp')):
            try:
                # Open the image
                with Image.open(file_path) as img:
                    # Check for alpha channel
                    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                        print(f"Fixing alpha channel: {file_path}")
                        img = img.convert('RGB')  # Convert to RGB
                        img.save(file_path)  # Save back without alpha channel
                        fixed_count += 1
                    # Verify image integrity
                    img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Removing corrupted file: {file_path}")
                os.remove(file_path)
                corrupted_count += 1
    return corrupted_count, fixed_count

def process_all_subdirectories(parent_directory):
    """
    Process all subdirectories in the parent directory to handle corrupted images
    and remove alpha channels.

    Args:
        parent_directory (str): Path to the parent directory.
    """
    total_corrupted = 0
    total_fixed = 0
    for sub_dir in os.listdir(parent_directory):
        sub_dir_path = os.path.join(parent_directory, sub_dir)
        if os.path.isdir(sub_dir_path):  # Ensure it's a directory
            print(f"Processing images in: {sub_dir_path}")
            corrupted_in_dir, fixed_in_dir = find_and_fix_images(sub_dir_path)
            print(f"Corrupted files removed from {sub_dir_path}: {corrupted_in_dir}")
            print(f"Images fixed (alpha channel removed) in {sub_dir_path}: {fixed_in_dir}")
            total_corrupted += corrupted_in_dir
            total_fixed += fixed_in_dir
    print(f"Total corrupted files removed: {total_corrupted}")
    print(f"Total images fixed (alpha channel removed): {total_fixed}")

# Start processing
process_all_subdirectories(parent_dir)


In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
# ==========================================
# PROFESSIONAL DATA DISTRIBUTION (AUTO SPLIT)
# 70% Train | 20% Val | 10% Test
# ==========================================

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# =============================
# CONFIG
# =============================
DATA_DIR = r"D:/M_Net_Diabetic/Dataset/5Class"
SEED = 111

np.random.seed(SEED)

# =============================
# LOAD DATA
# =============================
class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print("Detected Classes:", class_names)

paths, labels = [], []

for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for img in glob.glob(os.path.join(cls_path, "*")):
        if img.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(img)
            labels.append(cls)

paths = np.array(paths)
labels = np.array(labels)

print(f"\nTotal images: {len(paths)}")

# =============================
# SPLIT DATA (70 / 20 / 10)
# =============================
# Step 1: Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    paths, labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

# Step 2: Split Temp into Val (20%) and Test (10%)
# 20/30 = 0.6667
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1/3,
    stratify=y_temp,
    random_state=SEED
)

print("\nSplit Sizes:")
print(f"Train: {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test: {len(X_test)}")

# =============================
# COUNT FUNCTION
# =============================
def count_labels(labels, class_names):
    return np.array([np.sum(labels == c) for c in class_names])

train_counts = count_labels(y_train, class_names)
val_counts   = count_labels(y_val, class_names)
test_counts  = count_labels(y_test, class_names)

# =============================
# STYLE (PROFESSIONAL)
# =============================
sns.set(style="whitegrid", context="paper", font_scale=1.4)
palette = sns.color_palette("Set2", len(class_names))

# =============================
# BAR PLOTS (MAIN FIGURE)
# =============================
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

datasets = [
    ("Training Set", train_counts),
    ("Validation Set", val_counts),
    ("Test Set", test_counts),
]

for ax, (title, counts) in zip(axes, datasets):
    sns.barplot(x=class_names, y=counts, palette=palette, ax=ax)

    ax.set_title(title, fontsize=14, weight='bold')
    ax.set_xlabel("")
    ax.set_ylabel("Number of Images")
    ax.tick_params(axis='x', rotation=30)

    # Annotate values
    for i, v in enumerate(counts):
        ax.text(i, v + max(counts)*0.02, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig("data_distribution_bar.png", dpi=300)
plt.show()

# =============================
# PIE CHART (OVERALL ONLY)
# =============================
fig, ax = plt.subplots(figsize=(6, 6))

total_counts = train_counts + val_counts + test_counts

ax.pie(
    total_counts,
    labels=[c.title() for c in class_names],
    autopct='%1.1f%%',
    colors=palette,
    startangle=90,
    wedgeprops={'edgecolor': 'white'}
)

ax.set_title("Overall Class Distribution", fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig("data_distribution_pie.png", dpi=300)
plt.show()

# =============================
# SPLIT PIE (OPTIONAL CLEAN)
# =============================
fig, ax = plt.subplots(figsize=(6, 6))

sizes = [len(y_train), len(y_val), len(y_test)]
labels_split = ["Train", "Validation", "Test"]

ax.pie(
    sizes,
    labels=labels_split,
    autopct='%1.1f%%',
    colors=["#2a9d8f", "#e9c46a", "#e76f51"],
    startangle=90,
    explode=(0.05, 0.03, 0.03),
    wedgeprops={'edgecolor': 'white'}
)

ax.set_title("Dataset Split (70/20/10)", fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig("split_distribution.png", dpi=300)
plt.show()

# =============================
# TABLE (FOR PAPER)
# =============================
df = pd.DataFrame({
    "Class": class_names,
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts,
    "Total": total_counts
})

print("\n=== DATA DISTRIBUTION TABLE ===")
print(df.to_string(index=False))

df.to_csv("data_distribution_table.csv", index=False)

In [ ]:
# =============================
# FULL PIPELINE: CNN → FEATURES → GRAPHS → TABLE (FIXED)
# =============================

import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# =============================
# CONFIG
# =============================
DATA_DIR = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 123

np.random.seed(SEED)
tf.random.set_seed(SEED)

# =============================
# LOAD DATA
# =============================
class_names = sorted(os.listdir(DATA_DIR))
paths, labels = [], []

for i, c in enumerate(class_names):
    for p in glob.glob(os.path.join(DATA_DIR, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg")):
            paths.append(p)
            labels.append(i)

paths = np.array(paths)
labels = np.array(labels)

print(f"Loaded {len(paths)} images")

# =============================
# DATASET
# =============================

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)  # ✅ faster + fixed shape
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

ds = tf.data.Dataset.from_tensor_slices((paths, labels))
ds = ds.map(load_img).batch(BATCH_SIZE)

# =============================
# CNN FEATURE EXTRACTOR
# =============================
def build_cnn():
    return models.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu', name="feat")
    ])

cnn = build_cnn()
cnn.summary()

# =============================
# EXTRACT FEATURES
# =============================
features = []
for x, _ in ds:
    f = cnn(x, training=False).numpy()
    features.append(f)

X = np.vstack(features)
print("Feature shape:", X.shape)

# =============================
# STANDARDIZE
# =============================
scaler = StandardScaler()
X = scaler.fit_transform(X)

N = X.shape[0]

# =============================
# GRAPH BUILDERS (FIXED)
# =============================
def build_knn(X, k=10):
    N = X.shape[0]
    nn = NearestNeighbors(n_neighbors=k+1).fit(X)
    _, idx = nn.kneighbors(X)

    rows, cols = [], []
    for i in range(N):
        for j in idx[i][1:]:
            rows.append(i)
            cols.append(j)

    A = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))
    A = A.maximum(A.T)  # symmetric
    A.setdiag(1)
    return A.tocsr()

def build_epsilon(X, eps=1.0):
    N = X.shape[0]
    nn = NearestNeighbors(radius=eps).fit(X)
    ind = nn.radius_neighbors(X, return_distance=False)

    rows, cols = [], []
    for i, neighbors in enumerate(ind):
        for j in neighbors:
            if i != j:
                rows.append(i)
                cols.append(j)

    A = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))
    A = A.maximum(A.T)
    A.setdiag(1)
    return A.tocsr()

def build_rbf_sparse(X, gamma=0.01, k=12):
    """Sparse RBF using KNN (IMPORTANT FIX)"""
    N = X.shape[0]
    nn = NearestNeighbors(n_neighbors=k+1).fit(X)
    dist, idx = nn.kneighbors(X)

    rows, cols, data = [], [], []
    for i in range(N):
        for j, d in zip(idx[i][1:], dist[i][1:]):
            w = np.exp(-gamma * d * d)
            rows.append(i)
            cols.append(j)
            data.append(w)

    A = coo_matrix((data, (rows, cols)), shape=(N, N))
    A = A.maximum(A.T)
    A.setdiag(1)
    return A.tocsr()

def build_domain(paths):
    N = len(paths)
    groups = {}

    for i, p in enumerate(paths):
        key = os.path.basename(p).split("_")[0]
        groups.setdefault(key, []).append(i)

    rows, cols = [], []
    for g in groups.values():
        for i in g:
            for j in g:
                if i != j:
                    rows.append(i)
                    cols.append(j)

    A = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))
    A = A.maximum(A.T)
    A.setdiag(1)
    return A.tocsr()

# =============================
# BUILD GRAPHS
# =============================
A_knn = build_knn(X, k=12)
A_eps = build_epsilon(X, eps=1.5)
A_rbf = build_rbf_sparse(X, gamma=0.01, k=12)  # FIXED
A_dom = build_domain(paths)

# =============================
# GRAPH STATS (FIXED)
# =============================
def graph_stats(A, name):
    N = A.shape[0]

    # Remove self-loops safely
    A_no_diag = A.copy()
    A_no_diag.setdiag(0)
    A_no_diag.eliminate_zeros()

    # Edges
    edges = A_no_diag.nnz // 2

    # Degree
    deg = np.array(A_no_diag.sum(axis=1)).flatten()

    # Components
    comp, labels = connected_components(A_no_diag, directed=False)
    largest_cc = np.bincount(labels).max()

    density = edges / (N * (N - 1) / 2)

    return [
        name,
        N,
        edges,
        round(deg.mean(), 2),
        int(deg.min()),
        int(deg.max()),
        round(density, 6),
        comp,
        largest_cc
    ]

# =============================
# BUILD TABLE
# =============================
rows = [
    graph_stats(A_knn, "KNN"),
    graph_stats(A_eps, "ε-graph"),
    graph_stats(A_rbf, "RBF"),
    graph_stats(A_dom, "Domain")
]

df = pd.DataFrame(rows, columns=[
    "Graph","Nodes","Edges","AvgDeg","MinDeg","MaxDeg",
    "Density","Components","LargestCC"
])

print("\n=== GRAPH TABLE ===")
print(df.to_string(index=False))

# =============================
# SAVE
# =============================
df.to_csv("graph_stats.csv", index=False)

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> Hybrid based KNN graph building model

In [ ]:

# -----------------------
# [STEP 4] Extract 'feat' + (optional) PCA + standardize
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb, yb in ds:
        f = backbone(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_cnn, y_ordered = extract_features(all_ds)
assert np.all(y_ordered == labels_int), "Label order mismatch after feature extraction!"

if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True, svd_solver="auto")
    X = pca.fit_transform(X_cnn).astype(np.float32)
else:
    X = X_cnn

scaler = StandardScaler()
X_std = scaler.fit_transform(X).astype(np.float32)

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)
N, F = X_std.shape
print(f"[STEP 4] Features ready: N={N}, F={F} (backbone feat={X_cnn.shape[1]}; PCA={USE_PCA}, PCA_DIM={PCA_DIM})")

# [STATS A]
print(f"[STATS] Nodes (N) = {N}")
print(f"[STATS] Feature dimension (F) = {F}")

# -----------------------
# [STEP 5] Build mutual-kNN cosine graph
# -----------------------
nbrs = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(X_std)
dist, knn_idx = nbrs.kneighbors(X_std, return_distance=True)

rows, cols, data = [], [], []
for i in range(N):
    for j, d in zip(knn_idx[i], dist[i]):
        if i == j: 
            continue
        sim = 1.0 - float(d)
        if sim <= 0: continue
        rows.append(i); cols.append(j); data.append(sim)

A_dir = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
A_mut = A_dir.minimum(A_dir.T)  # reciprocal edges

# add self-loops
A_mut = A_mut.tolil()
A_mut.setdiag(1.0)
A_mut = A_mut.tocsr()

A_norm = gcn_filter(A_mut)

# [STATS B] edges & graph properties
nnz_total = A_mut.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2

degrees = np.asarray(A_mut.sum(axis=1)).ravel() - 1.0   # minus self-loop
avg_degree = degrees.mean()
max_degree = degrees.max()
min_degree = degrees.min()

possible_edges = N * (N - 1) / 2
density = undirected_edges / possible_edges if possible_edges > 0 else 0.0

from scipy.sparse.csgraph import connected_components
n_comp, labels_comp = connected_components(A_mut, directed=False)
largest_cc = np.bincount(labels_comp).max()

print(f"[STATS] Undirected edges (no self-loops) = {undirected_edges}")
print(f"[STATS] Avg degree (no self-loops) = {avg_degree:.2f} (min={min_degree:.0f}, max={max_degree:.0f})")
print(f"[STATS] Graph density = {density:.6f}")
print(f"[STATS] Connected components = {n_comp}; largest CC size = {largest_cc}")

# -----------------------
# [STEP 5A] Paper visuals
# -----------------------
# t-SNE (2D) — may take time; sample if very large
from sklearn.manifold import TSNE
print("[VIS] Running t-SNE (2D) on features...")
if N > 4000:
    # sample for speed
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(N, size=4000, replace=False)
    X_ts = X_std[sample_idx]
    y_ts = labels_int[sample_idx]
else:
    X_ts, y_ts = X_std, labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = (y_ts == i)
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=6, alpha=0.7, label=cname)
plt.title("t-SNE of Feature Embeddings")
plt.xlabel("t-SNE-1"); plt.ylabel("t-SNE-2")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.savefig("fig_tsne_features.png", dpi=300)
plt.show()

# Degree histogram
plt.figure(figsize=(7, 4))
plt.hist(degrees, bins=range(int(degrees.min()), int(degrees.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node degree (excluding self-loop)")
plt.ylabel("Count")
plt.title("Degree Distribution of Mutual-kNN Graph")
plt.tight_layout()
plt.savefig("fig_degree_hist.png", dpi=300)
plt.show()

# Adjacency snapshot (top-left block)
subN = min(150, N)
A_small = A_mut[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest")
plt.title(f"Adjacency (top-left {subN}×{subN})")
plt.xlabel("Node index"); plt.ylabel("Node index")
plt.tight_layout()
plt.savefig("fig_adjacency_snapshot.png", dpi=300)
plt.show()

# -----------------------
# [STEP 5B] OPTIONAL — one per-class image→pixel-graph (32×32)
# -----------------------
try:
    import cv2, networkx as nx
    from spektral.data import Graph as SpGraph
    px_image_size = (32, 32)

    def px_load_images(data_dir, image_size=(32, 32), classes=None):
        if classes is None:
            classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        imgs, lbls = [], []
        for cname in classes:
            cdir = os.path.join(data_dir, cname)
            if not os.path.isdir(cdir): 
                continue
            for f in os.listdir(cdir):
                p = os.path.join(cdir, f)
                img = cv2.imread(p, cv2.IMREAD_COLOR)
                if img is None:
                    continue
                if img.shape[-1] == 4:
                    img = img[:, :, :3]
                img = cv2.resize(img, image_size, interpolation=cv2.INTER_AREA)
                imgs.append(img); lbls.append(cname)
        name_to_id = {n: i for i, n in enumerate(classes)}
        lbls_int = np.array([name_to_id[s] for s in lbls], dtype=np.int32)
        return np.array(imgs), lbls_int, classes

    def image_to_graph(img):
        h, w, c = img.shape
        n = h * w
        x = (img.reshape(n, c).astype(np.float32)) / 255.0
        a = np.zeros((n, n), dtype=np.float32)
        for i in range(h):
            for j in range(w):
                idx = i * w + j
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < h and 0 <= nj < w:
                            a[idx, ni * w + nj] = 1.0
        return SpGraph(x=x, a=a)

    def visualize_image_and_graph(img_bgr, graph, title, savepath=None):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        G = nx.from_numpy_array(graph.a)
        pos = nx.spring_layout(G, seed=42)
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
        axes[0].imshow(img_rgb); axes[0].set_title(f"{title} — 32×32 image"); axes[0].axis("off")
        nx.draw(G, pos, node_size=12, node_color=graph.x[:, 0], cmap="viridis", with_labels=False, ax=axes[1])
        axes[1].set_title("8-neighbor pixel graph"); axes[1].axis("off")
        plt.tight_layout()
        if savepath: plt.savefig(savepath, dpi=300)
        plt.show()

    px_imgs, px_labels, px_classes = px_load_images(data_dir, image_size=px_image_size, classes=class_names)
    selected_idxs = []
    for cls_id, cname in enumerate(px_classes):
        idxs = np.where(px_labels == cls_id)[0]
        if idxs.size == 0:
            print(f"[WARN] No samples for '{cname}'"); 
            continue
        selected_idxs.append(int(idxs[0]))

    for i in selected_idxs:
        g = image_to_graph(px_imgs[i])
        cname = px_classes[px_labels[i]]
        visualize_image_and_graph(px_imgs[i], g, title=f"{cname}", savepath=f"fig_pixelgraph_oneperclass_{cname}.png")

except Exception as e:
    print(f"[STEP 5B] Skipped pixel-graph viz: {e}")

# -----------------------
# [STEP 6] Class balancing for GCN (optional)
# -----------------------
sample_w_train = mask_tr.astype(np.float32)
if USE_CLASS_BALANCING:
    counts = np.bincount(labels_int, minlength=num_classes).astype(np.float32)
    class_w = counts.sum() / np.maximum(counts, 1.0)
    class_w = class_w / class_w.mean()
    sample_w_train = sample_w_train * class_w[labels_int]



In [ ]:
# ==============================================
# KNN-only GCN pipeline with full evaluation
# ==============================================

import os, glob, time, re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from scipy.sparse import coo_matrix, csr_matrix
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# [STEP 0] Paths & params
# -----------------------
data_dir   = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# CNN training
LR_CNN, EPOCHS_CNN, AUGMENT = 1e-3, 25, True

# Feature PCA
USE_PCA, PCA_DIM = True, 512

# GCN & kNN
K_DEFAULT, LR_GCN, WD, EPOCHS_GCN = 12, 5e-3, 5e-4, 200
HIDDEN, DROPOUT, LABEL_SMOOTH = 64, 0.3, 0.05
VAL_SPLIT, TEST_SIZE = 0.2, 0.2
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# [STEP 1] Load dataset
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c:i for i,c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir,c)
    for p in glob.glob(os.path.join(cdir,"*")):
        if p.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):
            paths.append(p); labels_int.append(class_to_idx[c])

paths, labels_int = np.array(paths), np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes.")

# Train/val/test split
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr, mask_va, mask_te = np.zeros(len(paths),bool), np.zeros(len(paths),bool), np.zeros(len(paths),bool)
mask_tr[idx_tr], mask_va[idx_va], mask_te[idx_te] = True, True, True

# -----------------------
# [STEP 2] TF dataset
# -----------------------
def load_and_preprocess(path,label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape([IMG_SIZE[0],IMG_SIZE[1],3])
    return img, tf.cast(label,tf.int32)

def augment(img,label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.rot90(img, tf.random.uniform([],0,4,dtype=tf.int32))
    return img,label

def make_dataset(indexes,shuffle=True,batch_size=32,training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes],labels_int[indexes]))
    if shuffle: ds = ds.shuffle(len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True, batch_size=BATCH_SIZE, training=True)
val_ds = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE)
test_ds = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE)
all_ds = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE)

# -----------------------
# [STEP 3] CNN backbone
# -----------------------
def build_cnn_backbone(feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32,(4,4),activation="relu",input_shape=(IMG_SIZE[0],IMG_SIZE[1],3)),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(64,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat")
    ])

backbone = build_cnn_backbone()
inp = Input(shape=(IMG_SIZE[0],IMG_SIZE[1],3))
feat = backbone(inp)
out = layers.Dropout(0.5)(feat)
out = layers.Dense(num_classes, activation="softmax")(out)
cnn = Model(inp,out)
cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
        callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
        verbose=1)

# -----------------------
# [STEP 4] Extract features
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb,yb in ds:
        feats.append(backbone(xb,training=False).numpy())
        ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_tr,_ = extract_features(make_dataset(idx_tr,shuffle=False,batch_size=BATCH_SIZE))
X_va,_ = extract_features(make_dataset(idx_va,shuffle=False,batch_size=BATCH_SIZE))
X_te,_ = extract_features(make_dataset(idx_te,shuffle=False,batch_size=BATCH_SIZE))

if USE_PCA:
    pca = PCA(n_components=PCA_DIM,random_state=SEED,whiten=True)
    pca.fit(X_tr)
    X_tr, X_va, X_te = pca.transform(X_tr), pca.transform(X_va), pca.transform(X_te)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)

F = X_tr_s.shape[1]
X_std = np.zeros((len(paths),F),dtype=np.float32)
X_std[idx_tr] = X_tr_s
X_std[idx_va] = X_va_s
X_std[idx_te] = X_te_s

Y_all = to_categorical(labels_int,num_classes=num_classes).astype(np.float32)
Y_train_only = np.zeros_like(Y_all)
Y_train_only[mask_tr] = Y_all[mask_tr]

# -----------------------
# [STEP 5] kNN graph only
# -----------------------
def build_graph_knn_cosine(X,k=12):
    N = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(X)
    dist, idx = nbrs.kneighbors(X)
    rows,cols,data = [],[],[]
    for i in range(N):
        for j,d in zip(idx[i], dist[i]):
            if i==j: continue
            sim = 1.0 - float(d)
            if sim <= 0: continue
            rows.append(i); cols.append(j); data.append(sim)
    A = coo_matrix((data,(rows,cols)),shape=(N,N)).minimum(coo_matrix((data,(cols,rows)),shape=(N,N)))
    A.setdiag(1.0)
    return gcn_filter(A.tocsr())

A_knn_norm = build_graph_knn_cosine(X_std,k=K_DEFAULT)

# -----------------------
# [STEP 6] GCN
# -----------------------
def build_gcn(F_dim,N_nodes):
    X_in = Input(shape=(F_dim,))
    A_in = Input((N_nodes,),sparse=True)
    h = GCNConv(HIDDEN,activation=None,kernel_regularizer=regularizers.l2(WD))([X_in,A_in])
    h = layers.BatchNormalization()(h)
    h = layers.Activation("relu")(h)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.Add()([h,layers.Dense(HIDDEN,use_bias=False)(X_in)])
    logits = GCNConv(num_classes,activation=None)([h,A_in])
    out = layers.Activation("softmax")(logits)
    model = Model([X_in,A_in],out)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  weighted_metrics=["accuracy"])
    return model

gcn_model = build_gcn(F, X_std.shape[0])

# -----------------------
# [STEP 7] Train GCN
# -----------------------
history = gcn_model.fit([X_std,A_knn_norm], Y_train_only, sample_weight=None,
                        batch_size=X_std.shape[0], epochs=EPOCHS_GCN, shuffle=False,
                        validation_data=([X_std,A_knn_norm],Y_all,mask_va.astype(np.float32)),
                        callbacks=[EarlyStopping(monitor="val_loss",patience=20,restore_best_weights=True)],
                        verbose=1)

# -----------------------
# [STEP 8] Evaluate & plot metrics
# -----------------------
y_prob = gcn_model.predict([X_std,A_knn_norm],batch_size=X_std.shape[0])
y_pred = np.argmax(y_prob,axis=1)

def plot_cm_roc(y_true_mask,label):
    y_true = labels_int[y_true_mask]
    y_pred_mask = y_pred[y_true_mask]
    print(f"\n=== {label} Classification Report ===")
    print(classification_report(y_true,y_pred_mask,target_names=class_names))
    cm = confusion_matrix(y_true,y_pred_mask)
    plt.figure(figsize=(6,6))
    plt.imshow(cm,cmap="Blues")
    plt.title(f"{label} Confusion Matrix")
    plt.colorbar()
    plt.xticks(range(num_classes),class_names,rotation=45)
    plt.yticks(range(num_classes),class_names)
    for i in range(num_classes):
        for j in range(num_classes): plt.text(j,i,cm[i,j],ha="center",va="center",color="red")
    plt.tight_layout(); plt.show()
    # ROC
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    y_prob_bin = y_prob[y_true_mask]
    plt.figure(figsize=(8,6))
    for i in range(num_classes):
        fpr,tpr,_ = roc_curve(y_bin[:,i],y_prob_bin[:,i])
        plt.plot(fpr,tpr,label=f"{class_names[i]}")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{label} ROC"); plt.legend(); plt.show()

plot_cm_roc(mask_va,"Validation")
plot_cm_roc(mask_te,"Test")

# Loss & accuracy curves
plt.figure(); plt.plot(history.history['loss'],label='Train Loss'); plt.plot(history.history['val_loss'],label='Val Loss')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("GCN Loss"); plt.legend(); plt.show()

plt.figure(); plt.plot(history.history['accuracy'],label='Train Acc'); plt.plot(history.history['val_accuracy'],label='Val Acc')
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("GCN Accuracy"); plt.legend(); plt.show()

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 2 <span style='color:blue'>|</span> Hybrid ε-graph building model

In [ ]:
# ==============================
# ε-Graph Only GCN Pipeline
# ==============================

import os, glob, time
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.neighbors import NearestNeighbors

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from scipy.sparse import coo_matrix
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# Paths & Params
# -----------------------
data_dir = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123
LR_CNN = 1e-3
EPOCHS_CNN = 25
AUGMENT = True
USE_PCA = True
PCA_DIM = 512
LR_GCN = 5e-3
WD = 5e-4
EPOCHS_GCN = 200
HIDDEN = 64
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT = 0.2
TEST_SIZE = 0.2
USE_CLASS_BALANCING = True
EPSILON = 0.20

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# Load dataset
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir,d))])
class_to_idx = {c:i for i,c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    for p in glob.glob(os.path.join(data_dir,c,"*")):
        if p.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])
paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

# Train/val/test split
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])
mask_tr, mask_va, mask_te = np.zeros(len(paths),bool), np.zeros(len(paths),bool), np.zeros(len(paths),bool)
mask_tr[idx_tr] = True; mask_va[idx_va] = True; mask_te[idx_te] = True

# -----------------------
# TF Dataset
# -----------------------
def load_and_preprocess(path,label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)

def augment(img,label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.rot90(img, tf.random.uniform([],0,4,dtype=tf.int32))
    return img,label

def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle: ds = ds.shuffle(len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True, batch_size=BATCH_SIZE, training=True)
val_ds = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE)
test_ds = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE)

# -----------------------
# CNN backbone
# -----------------------
def build_cnn_backbone(feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32,(4,4),activation="relu",input_shape=(IMG_SIZE[0],IMG_SIZE[1],3)),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(64,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat")
    ])
backbone = build_cnn_backbone()
inp = Input(shape=(IMG_SIZE[0],IMG_SIZE[1],3))
feat = backbone(inp)
out = layers.Dropout(0.5)(feat)
out = layers.Dense(num_classes, activation="softmax")(out)
cnn = Model(inp,out)
cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
        callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
        verbose=1)

# -----------------------
# Extract features + PCA + Standardize
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb,yb in ds:
        feats.append(backbone(xb, training=False).numpy())
        ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_tr,_ = extract_features(make_dataset(idx_tr, shuffle=False, batch_size=BATCH_SIZE))
X_va,_ = extract_features(make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE))
X_te,_ = extract_features(make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE))

if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True)
    pca.fit(X_tr)
    X_tr, X_va, X_te = pca.transform(X_tr), pca.transform(X_va), pca.transform(X_te)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)

F = X_tr_s.shape[1]
X_std = np.zeros((len(paths),F),dtype=np.float32)
X_std[idx_tr] = X_tr_s
X_std[idx_va] = X_va_s
X_std[idx_te] = X_te_s

Y_all = to_categorical(labels_int,num_classes=num_classes)
Y_train_only = np.zeros_like(Y_all)
Y_train_only[mask_tr] = Y_all[mask_tr]

# -----------------------
# ε-Graph builder only
# -----------------------
def build_graph_epsilon_cosine(X, eps=EPSILON):
    radius = max(1e-6, 1.0 - float(eps))
    N = X.shape[0]
    nn = NearestNeighbors(metric="cosine", radius=radius).fit(X)
    nbrs = nn.radius_neighbors(X, return_distance=False)
    rows, cols, data = [], [], []
    for i,n in enumerate(nbrs):
        for j in n:
            if i==j: continue
            rows.append(i); cols.append(j); data.append(1.0)
    A = coo_matrix((data,(rows,cols)), shape=(N,N)).minimum(coo_matrix((data,(cols,rows)), shape=(N,N)))
    A.setdiag(1.0)
    return gcn_filter(A.tocsr())

A_eps_norm = build_graph_epsilon_cosine(X_std)

# -----------------------
# GCN
# -----------------------
def build_gcn(F_dim,N_nodes):
    X_in = Input(shape=(F_dim,))
    A_in = Input((N_nodes,), sparse=True)
    h = GCNConv(HIDDEN,activation=None,kernel_regularizer=regularizers.l2(WD))([X_in,A_in])
    h = layers.BatchNormalization()(h)
    h = layers.Activation("relu")(h)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.Add()([h, layers.Dense(HIDDEN,use_bias=False)(X_in)])
    logits = GCNConv(num_classes,activation=None)([h,A_in])
    out = layers.Activation("softmax")(logits)
    model = Model([X_in,A_in],out)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  weighted_metrics=["accuracy"])
    return model

gcn_model = build_gcn(F, X_std.shape[0])

# -----------------------
# Train GCN
# -----------------------
history = gcn_model.fit([X_std,A_eps_norm], Y_train_only,
                        batch_size=X_std.shape[0], epochs=EPOCHS_GCN, shuffle=False,
                        validation_data=([X_std,A_eps_norm],Y_all,mask_va.astype(np.float32)),
                        callbacks=[EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)],
                        verbose=1)

# -----------------------
# Evaluate & plots
# -----------------------
y_prob = gcn_model.predict([X_std,A_eps_norm], batch_size=X_std.shape[0])
y_pred = np.argmax(y_prob, axis=1)

def plot_cm_roc(mask,label):
    y_true = labels_int[mask]
    y_pred_mask = y_pred[mask]
    print(f"\n=== {label} Classification Report ===")
    print(classification_report(y_true, y_pred_mask, target_names=class_names))
    # Confusion
    cm = confusion_matrix(y_true, y_pred_mask)
    plt.figure(figsize=(6,6))
    plt.imshow(cm,cmap="Blues")
    plt.title(f"{label} Confusion Matrix")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45)
    plt.yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j,i,cm[i,j],ha="center",va="center",color="red")
    plt.tight_layout(); plt.show()
    # ROC
    y_bin = label_binarize(y_true, classes=np.arange(num_classes))
    y_prob_mask = y_prob[mask]
    plt.figure(figsize=(8,6))
    for i in range(num_classes):
        fpr,tpr,_ = roc_curve(y_bin[:,i],y_prob_mask[:,i])
        plt.plot(fpr,tpr,label=class_names[i])
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{label} ROC"); plt.legend(); plt.show()

plot_cm_roc(mask_va,"Validation")
plot_cm_roc(mask_te,"Test")

# Loss & accuracy curves
plt.figure(); plt.plot(history.history['loss'],label='Train Loss'); plt.plot(history.history['val_loss'],label='Val Loss')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("GCN Loss"); plt.legend(); plt.show()
plt.figure(); plt.plot(history.history['accuracy'],label='Train Acc'); plt.plot(history.history['val_accuracy'],label='Val Acc')
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("GCN Accuracy"); plt.legend(); plt.show()

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 3 <span style='color:blue'>|</span> RBF 

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.neighbors import NearestNeighbors

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
from scipy.sparse import coo_matrix, csr_matrix

# -----------------------
# [STEP 0] Paths & params
# -----------------------
data_dir   = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

LR_CNN     = 1e-3
EPOCHS_CNN = 25
AUGMENT    = True

USE_PCA    = True
PCA_DIM    = 512
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2
USE_CLASS_BALANCING = True

# RBF/GCN params
K_DEFAULT  = 12        # number of nearest neighbors for RBF graph
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# [STEP 1] Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c:i for i,c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])
paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes: {class_names}")

all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])
mask_tr, mask_va, mask_te = np.zeros(len(paths),bool), np.zeros(len(paths),bool), np.zeros(len(paths),bool)
mask_tr[idx_tr] = True; mask_va[idx_va] = True; mask_te[idx_te] = True

train_labels, val_labels, test_labels = labels_int[idx_tr], labels_int[idx_va], labels_int[idx_te]

# -----------------------
# [STEP 2] Dataset functions
# -----------------------
def load_and_preprocess(path,label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img,channels=3,expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape([IMG_SIZE[0],IMG_SIZE[1],3])
    return img, tf.cast(label,tf.int32)

def augment(img,label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([],0,4,dtype=tf.int32)
        img = tf.image.rot90(img,k)
    return img,label

def make_dataset(indexes,shuffle=True,batch_size=32,training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(len(indexes),seed=SEED,reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr,shuffle=True,batch_size=BATCH_SIZE,training=True)
val_ds   = make_dataset(idx_va,shuffle=False,batch_size=BATCH_SIZE)
test_ds  = make_dataset(idx_te,shuffle=False,batch_size=BATCH_SIZE)

# -----------------------
# [STEP 3] CNN Backbone
# -----------------------
def build_cnn_backbone(input_shape=(IMG_SIZE[0],IMG_SIZE[1],3),feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32,(4,4),activation="relu",input_shape=input_shape),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(64,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.MaxPooling2D((3,3)),
        layers.Conv2D(128,(4,4),activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim,activation="relu",name="feat")
    ])

backbone = build_cnn_backbone()
inp = Input(shape=(IMG_SIZE[0],IMG_SIZE[1],3))
feat = backbone(inp)
out = layers.Dropout(0.5)(feat)
out = layers.Dense(num_classes,activation="softmax")(out)
cnn = Model(inp,out)
cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
        callbacks=[EarlyStopping(monitor="val_loss",patience=8,restore_best_weights=True),
                   ModelCheckpoint("best_cnn.keras",monitor="val_loss",save_best_only=True)],
        verbose=1)

# -----------------------
# [STEP 4] Extract features
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb,yb in ds:
        feats.append(backbone(xb,training=False).numpy())
        ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_tr_raw,_ = extract_features(make_dataset(idx_tr,shuffle=False,batch_size=BATCH_SIZE))
X_va_raw,_ = extract_features(make_dataset(idx_va,shuffle=False,batch_size=BATCH_SIZE))
X_te_raw,_ = extract_features(make_dataset(idx_te,shuffle=False,batch_size=BATCH_SIZE))

if USE_PCA:
    pca = PCA(n_components=PCA_DIM,random_state=SEED,whiten=True)
    pca.fit(X_tr_raw)
    X_tr = pca.transform(X_tr_raw).astype(np.float32)
    X_va = pca.transform(X_va_raw).astype(np.float32)
    X_te = pca.transform(X_te_raw).astype(np.float32)
else:
    X_tr, X_va, X_te = X_tr_raw, X_va_raw, X_te_raw

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr).astype(np.float32)
X_va_s = scaler.transform(X_va).astype(np.float32)
X_te_s = scaler.transform(X_te).astype(np.float32)

F = X_tr_s.shape[1]
N = len(paths)
X_std = np.zeros((N,F),dtype=np.float32)
X_std[idx_tr]=X_tr_s
X_std[idx_va]=X_va_s
X_std[idx_te]=X_te_s

Y_all = to_categorical(labels_int,num_classes)
Y_train_only = np.zeros_like(Y_all)
Y_train_only[mask_tr]=Y_all[mask_tr]

# -----------------------
# [STEP 5] Class balancing
# -----------------------
sample_w_train=None
if USE_CLASS_BALANCING:
    counts_tr=np.bincount(labels_int[mask_tr],minlength=num_classes).astype(np.float32)
    class_w=counts_tr.sum()/np.maximum(counts_tr,1)
    class_w=class_w/class_w.mean()
    sample_w_train=np.zeros(N,dtype=np.float32)
    sample_w_train[mask_tr]=class_w[labels_int[mask_tr]]

# -----------------------
# [STEP 6] RBF graph only
# -----------------------
def build_graph_rbf_kernel(X,gamma=1.0,use_knn=True,k=12):
    N_local=X.shape[0]
    if use_knn:
        nn=NearestNeighbors(n_neighbors=k+1,metric="euclidean").fit(X)
        dist,knn_idx=nn.kneighbors(X,return_distance=True)
        rows,cols,data=[],[],[]
        for i in range(N_local):
            for j,d in zip(knn_idx[i],dist[i]):
                if i==j: continue
                w=np.exp(-gamma*float(d)*float(d))
                if w<=0: continue
                rows.append(i); cols.append(j); data.append(w)
        A_dir=coo_matrix((data,(rows,cols)),shape=(N_local,N_local),dtype=np.float32)
        A=A_dir.maximum(A_dir.T).tolil()
        A.setdiag(1.0)
        return A.tocsr()
    else:
        from sklearn.metrics.pairwise import euclidean_distances
        D=euclidean_distances(X,X,squared=True).astype(np.float32)
        W=np.exp(-gamma*D)
        np.fill_diagonal(W,1.0)
        return csr_matrix(W)

def make_A_norm_rbf(gamma=None,k=None):
    if gamma is None: gamma=1.0/max(1,F)
    if k is None: k=12
    return gcn_filter(build_graph_rbf_kernel(X_std,gamma=gamma,use_knn=True,k=k))

# -----------------------
# [STEP 7] GCN builder
# -----------------------
def build_gcn_model(F_dim,N_nodes):
    X_in=Input(shape=(F_dim,))
    A_in=Input((N_nodes,),sparse=True)
    h1=GCNConv(HIDDEN,activation=None,kernel_regularizer=regularizers.l2(WD))([X_in,A_in])
    h1=layers.BatchNormalization()(h1)
    h1=layers.Activation("relu")(h1)
    h1=layers.Dropout(DROPOUT)(h1)
    res=layers.Dense(HIDDEN,use_bias=False)(X_in)
    h1=layers.Add()([h1,res])
    logits=GCNConv(num_classes,activation=None)([h1,A_in])
    out=layers.Activation("softmax")(logits)
    model=Model([X_in,A_in],out)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  weighted_metrics=["accuracy"])
    return model

# -----------------------
# [STEP 8] Train GCN (RBF only)
# -----------------------
gamma_base = 1.0/max(1,F)
A_norm_rbf = make_A_norm_rbf(gamma=gamma_base,k=K_DEFAULT)
model_gcn = build_gcn_model(F,N)

history = model_gcn.fit([X_std,A_norm_rbf],Y_train_only,
                        sample_weight=sample_w_train,
                        batch_size=N,
                        epochs=EPOCHS_GCN,
                        shuffle=False,
                        validation_data=([X_std,A_norm_rbf],Y_all,mask_va.astype(np.float32)),
                        callbacks=[EarlyStopping(monitor="val_loss",patience=20,restore_best_weights=True),
                                   ModelCheckpoint("best_gcn_rbf.keras",monitor="val_loss",save_best_only=True)],
                        verbose=1)

# -----------------------
# [STEP 9] Evaluation
# -----------------------
from sklearn.preprocessing import label_binarize

y_prob_all=model_gcn.predict([X_std,A_norm_rbf],batch_size=N,verbose=0)
y_pred_all=np.argmax(y_prob_all,axis=1)

# Classification report
print("\nValidation Classification Report:\n")
print(classification_report(labels_int[mask_va],y_pred_all[mask_va],target_names=class_names))
print("\nTest Classification Report:\n")
print(classification_report(labels_int[mask_te],y_pred_all[mask_te],target_names=class_names))

# Confusion matrix function
def plot_confusion(cm,class_names,title="Confusion Matrix"):
    plt.figure(figsize=(6,6))
    plt.imshow(cm,cmap="Blues")
    plt.title(title)
    plt.colorbar()
    plt.xticks(range(len(class_names)),class_names,rotation=45)
    plt.yticks(range(len(class_names)),class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j,i,cm[i,j],ha="center",va="center",color="red")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

cm_val = confusion_matrix(labels_int[mask_va],y_pred_all[mask_va])
cm_te  = confusion_matrix(labels_int[mask_te],y_pred_all[mask_te])
plot_confusion(cm_val,class_names,"Validation Confusion Matrix")
plot_confusion(cm_te,class_names,"Test Confusion Matrix")

# ROC curves
def plot_roc(y_true,y_prob,class_names,title="ROC Curves"):
    y_bin = label_binarize(y_true,classes=np.arange(len(class_names)))
    fpr,tpr,roc_auc = {},{},{}
    for i in range(len(class_names)):
        fpr[i],tpr[i],_ = roc_curve(y_bin[:,i],y_prob[:,i])
        roc_auc[i]=auc(fpr[i],tpr[i])
    fpr["micro"],tpr["micro"],_ = roc_curve(y_bin.ravel(),y_prob.ravel())
    roc_auc["micro"]=auc(fpr["micro"],tpr["micro"])
    plt.figure(figsize=(8,6))
    for i in range(len(class_names)):
        plt.plot(fpr[i],tpr[i],label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
    plt.plot(fpr["micro"],tpr["micro"],linestyle="--",label=f"Micro (AUC={roc_auc['micro']:.2f})")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

plot_roc(labels_int[mask_va],y_prob_all[mask_va],class_names,"Validation ROC Curves")
plot_roc(labels_int[mask_te],y_prob_all[mask_te],class_names,"Test ROC Curves")

# Accuracy and Loss Curves
def plot_training_curves(history):
    acc = history.history['accuracy']
    val_acc = history.history.get('val_accuracy',None)
    loss = history.history['loss']
    val_loss = history.history.get('val_loss',None)
    epochs = range(1,len(acc)+1)
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.plot(epochs,acc,'b-',label='Train Acc')
    if val_acc is not None: plt.plot(epochs,val_acc,'r-',label='Val Acc')
    plt.title("Accuracy Curve"); plt.xlabel("Epochs"); plt.ylabel("Accuracy"); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(epochs,loss,'b-',label='Train Loss')
    if val_loss is not None: plt.plot(epochs,val_loss,'r-',label='Val Loss')
    plt.title("Loss Curve"); plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend()
    plt.show()

plot_training_curves(history)

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 4 <span style='color:blue'>|</span> Domain 

In [ ]:

import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
from scipy.sparse import coo_matrix

# -----------------------
# [STEP 0] Paths & params
# -----------------------
data_dir   = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# CNN params
LR_CNN     = 1e-3
EPOCHS_CNN = 25
AUGMENT    = True

# PCA / GCN
USE_PCA    = True
PCA_DIM    = 512
K_DEFAULT  = 12
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# [STEP 1] Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths      = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes: {class_names}")

# Stratified splits
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
mask_te = np.zeros(len(paths), dtype=bool); mask_te[idx_te] = True

train_labels = labels_int[idx_tr]
val_labels   = labels_int[idx_va]
test_labels  = labels_int[idx_te]

# -----------------------
# [STEP 2] TF dataset
# -----------------------
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)

def augment(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label

def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE, training=True)
val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE)
test_ds  = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE)
all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE)

# -----------------------
# [STEP 3] CNN backbone
# -----------------------
def build_cnn_backbone(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=input_shape),
        layers.MaxPooling2D((3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu"),
        layers.MaxPooling2D((3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.MaxPooling2D((3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat"),
    ])

backbone = build_cnn_backbone()
inp  = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
feat = backbone(inp)
out  = layers.Dropout(0.5)(feat)
out  = layers.Dense(num_classes, activation="softmax")(out)
cnn  = Model(inp, out)

cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
        callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                   ModelCheckpoint("best_cnn.keras", monitor="val_loss", save_best_only=True)],
        verbose=1)

# -----------------------
# [STEP 4] Extract features
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb, yb in ds:
        feats.append(backbone(xb, training=False).numpy())
        ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_tr_raw, _ = extract_features(make_dataset(idx_tr, shuffle=False, batch_size=BATCH_SIZE))
X_va_raw, _ = extract_features(make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE))
X_te_raw, _ = extract_features(make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE))

if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True)
    pca.fit(X_tr_raw)
    X_tr = pca.transform(X_tr_raw)
    X_va = pca.transform(X_va_raw)
    X_te = pca.transform(X_te_raw)
else:
    X_tr, X_va, X_te = X_tr_raw, X_va_raw, X_te_raw

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)

F = X_tr_s.shape[1]
N = len(paths)
X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr_s
X_std[idx_va] = X_va_s
X_std[idx_te] = X_te_s

Y_all = to_categorical(labels_int, num_classes)
Y_train_only = np.zeros_like(Y_all)
Y_train_only[mask_tr] = Y_all[mask_tr]

# -----------------------
# [STEP 5] Class balancing
# -----------------------
sample_w_train = None
if USE_CLASS_BALANCING:
    counts_tr = np.bincount(labels_int[mask_tr], minlength=num_classes).astype(np.float32)
    class_w = counts_tr.sum() / np.maximum(counts_tr, 1)
    class_w = class_w / class_w.mean()
    sample_w_train = np.zeros(N, dtype=np.float32)
    sample_w_train[mask_tr] = class_w[labels_int[mask_tr]]

# -----------------------
# [STEP 6] Robust domain graph
# -----------------------
def build_graph_domain_rules(paths_list, class_names=class_names):
    N = len(paths_list)
    class_to_indices = {c: [] for c in class_names}
    for idx, p in enumerate(paths_list):
        cls = os.path.basename(os.path.dirname(p))
        if cls not in class_to_indices:
            cls = class_names[0]
        class_to_indices[cls].append(idx)
    rows, cols, data = [], [], []
    for idxs in class_to_indices.values():
        for i in idxs:
            for j in idxs:
                rows.append(i)
                cols.append(j)
                data.append(1.0)
    A_dir = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
    A = A_dir.maximum(A_dir.T).tolil()
    A.setdiag(1.0)
    return A.tocsr()

def make_A_norm(**kwargs):
    return gcn_filter(build_graph_domain_rules(paths))

# -----------------------
# [STEP 7] GCN builder
# -----------------------
def build_gcn_model(F_dim, N_nodes):
    X_in = Input(shape=(F_dim,), name="X_in")
    A_in = Input((N_nodes,), sparse=True, name="A_in")
    h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD))([X_in, A_in])
    h1 = layers.BatchNormalization()(h1)
    h1 = layers.Activation("relu")(h1)
    h1 = layers.Dropout(DROPOUT)(h1)
    res = layers.Dense(HIDDEN, use_bias=False)(X_in)
    h1 = layers.Add()([h1, res])
    logits = GCNConv(num_classes, activation=None)([h1, A_in])
    out = layers.Activation("softmax")(logits)
    model = Model(inputs=[X_in, A_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  weighted_metrics=["accuracy"])
    return model

# -----------------------
# [STEP 8] Train GCN (Domain)
# -----------------------
A_norm_local = make_A_norm()
model_gcn = build_gcn_model(F, N)

print("X_std shape:", X_std.shape)
print("Y_train_only shape:", Y_train_only.shape)
print("A_norm_local shape:", A_norm_local.shape)
print("sample_w_train shape:", sample_w_train.shape if sample_w_train is not None else None)

history = model_gcn.fit(
    [X_std, A_norm_local],
    Y_train_only,
    sample_weight=sample_w_train,
    batch_size=N,
    epochs=EPOCHS_GCN,
    shuffle=False,
    validation_data=([X_std, A_norm_local], Y_all, mask_va.astype(np.float32)),
    callbacks=[EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
               ModelCheckpoint("best_gcn_domain.keras", monitor="val_loss", save_best_only=True)],
    verbose=1
)

# -----------------------
# [STEP 9] Evaluate
# -----------------------
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
# -----------------------
# [STEP 10] Predictions
# -----------------------
y_prob_all = model_gcn.predict([X_std, A_norm_local], batch_size=N)
y_pred_all = np.argmax(y_prob_all, axis=1)

# -----------------------
# Utility Functions
# -----------------------
def plot_confusion(cm, class_names, title="Confusion Matrix"):
    plt.figure(figsize=(6,6))
    plt.imshow(cm, cmap='Blues')
    plt.title(title)
    plt.colorbar()
    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='red')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def plot_roc(y_true, y_prob, class_names, title="ROC Curves"):
    y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    # Micro-average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_prob.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    # Plot
    plt.figure(figsize=(8,6))
    for i in range(len(class_names)):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
    plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

def print_classification_report(y_true, y_pred, class_names, title="Classification Report"):
    print(f"\n==== {title} ====\n")
    report = classification_report(y_true, y_pred, target_names=class_names)
    print(report)

# -----------------------
# Validation Evaluation
# -----------------------
mask = mask_va
y_true_val = labels_int[mask]
y_pred_val = y_pred_all[mask]
y_prob_val = y_prob_all[mask]

print_classification_report(y_true_val, y_pred_val, class_names, title="Validation Classification Report")

cm_val = confusion_matrix(y_true_val, y_pred_val)
plot_confusion(cm_val, class_names, title="Validation Confusion Matrix")
plot_roc(y_true_val, y_prob_val, class_names, title="Validation ROC Curves")

# -----------------------
# Test Evaluation
# -----------------------
mask = mask_te
y_true_te = labels_int[mask]
y_pred_te = y_pred_all[mask]
y_prob_te = y_prob_all[mask]

print_classification_report(y_true_te, y_pred_te, class_names, title="Test Classification Report")

cm_te = confusion_matrix(y_true_te, y_pred_te)
plot_confusion(cm_te, class_names, title="Test Confusion Matrix")
plot_roc(y_true_te, y_prob_te, class_names, title="Test ROC Curves")

# -----------------------
# Training Curves
# -----------------------
def plot_training_curves(history):
    acc = history.history['accuracy']
    val_acc = history.history.get('val_accuracy', None)
    loss = history.history['loss']
    val_loss = history.history.get('val_loss', None)
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(14,5))
    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(epochs, acc, 'b-', label='Train Acc')
    if val_acc is not None:
        plt.plot(epochs, val_acc, 'r-', label='Val Acc')
    plt.title("Accuracy Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1,2,2)
    plt.plot(epochs, loss, 'b-', label='Train Loss')
    if val_loss is not None:
        plt.plot(epochs, val_loss, 'r-', label='Val Loss')
    plt.title("Loss Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

# Plot CNN training curves (optional)
# plot_training_curves(hist_cnn)  # if you kept CNN history
# Plot GCN training curves
plot_training_curves(history)
val_loss, val_acc = model_gcn.evaluate([X_std, A_norm_local], Y_all, sample_weight=mask_va.astype(np.float32), batch_size=N)
test_loss, test_acc = model_gcn.evaluate([X_std, A_norm_local], Y_all, sample_weight=mask_te.astype(np.float32), batch_size=N)

print(f"Domain GCN - Validation acc: {val_acc:.4f}, Test acc: {test_acc:.4f}")

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 5 <span style='color:blue'>|</span> CNN based all graph building models

In [ ]:
# =======================================
# CNN (up to 'feat') -> Graph builders -> GCN
# with tiny grid search over K, ε, γ
# + results table (best per method) & best-method CM + ROC
# =======================================

import os, glob, time, re
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# [STEP 0] Paths & params
# -----------------------
data_dir   = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# CNN training params
LR_CNN     = 1e-3
EPOCHS_CNN = 25
AUGMENT    = True

# Feature post-processing
USE_PCA    = True
PCA_DIM    = 512   # try 256/512

# Graph/GCN params (base defaults; grid search will override)
K_DEFAULT  = 12
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# [STEP 1] Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p); labels_int.append(class_to_idx[c])

paths      = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes: {class_names}")

# Stratified splits -> masks
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
mask_te = np.zeros(len(paths), dtype=bool); mask_te[idx_te] = True

# For split pies later
train_labels = labels_int[idx_tr]
val_labels   = labels_int[idx_va]
test_labels  = labels_int[idx_te]

# -----------------------
# [STEP 2] TF dataset
# -----------------------
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)     # [0,1]
    img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)

def augment(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label

def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training and AUGMENT:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE, training=True)
val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE, training=False)
test_ds  = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE, training=False)
all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE, training=False)

# -----------------------
# [STEP 2A] Split pies (dataset composition)
# -----------------------
def plot_split_pies():
    CLASS_TYPES = class_names
    N_TYPES = len(CLASS_TYPES)
    def counts_per(labels_vec): return np.bincount(labels_vec, minlength=N_TYPES)

    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(20, 14))
    ax = ax.flatten()

    # Train/Val sizes
    ax[0].set_title('Train–Validation Split', fontsize=22)
    ax[0].pie([len(train_labels), len(val_labels)], labels=['Train','Validation'],
              colors=['darkcyan','orange'],
              autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p*(len(train_labels)+len(val_labels))/100),
              explode=(0.1, 0), startangle=85, textprops={'fontsize':20})

    # Train per-class
    train_cnt = counts_per(train_labels)
    print('Training Counts per class:', dict(zip(CLASS_TYPES, train_cnt)))
    ax[1].set_title('Training Data (per class)', fontsize=22)
    ax[1].pie(train_cnt, labels=[c.title() for c in CLASS_TYPES],
              colors=['#FAC500','#0BFA00','#0066FA','#FA0000'][:N_TYPES],
              autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p*train_cnt.sum()/100),
              explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize':20})

    # Val per-class
    val_cnt = counts_per(val_labels)
    print('Validation Counts per class:', dict(zip(CLASS_TYPES, val_cnt)))
    ax[2].set_title('Validation Data (per class)', fontsize=22)
    ax[2].pie(val_cnt, labels=[c.title() for c in CLASS_TYPES],
              colors=['#FAC500','#0BFA00','#0066FA','#FA0000'][:N_TYPES],
              autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p*val_cnt.sum()/100),
              explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize':20})

    # Test per-class
    test_cnt = counts_per(test_labels)
    print('Testing Counts per class:', dict(zip(CLASS_TYPES, test_cnt)))
    ax[3].set_title('Testing Data (per class)', fontsize=22)
    ax[3].pie(test_cnt, labels=[c.title() for c in CLASS_TYPES],
              colors=['#FAC500','#0BFA00','#0066FA','#FA0000'][:N_TYPES],
              autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p*test_cnt.sum()/100),
              explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize':20})

    plt.tight_layout()
    plt.savefig("fig_split_pies.png", dpi=300)
    plt.show()

plot_split_pies()

# -----------------------
# [STEP 3] CNN backbone that ENDS at 'feat'
# -----------------------
def build_cnn_backbone(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat"),  # <-- stops here
    ])

backbone = build_cnn_backbone()

# Temporary head for supervised CNN training
inp  = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
feat = backbone(inp)
out  = layers.Dropout(0.5)(feat)
out  = layers.Dense(num_classes, activation="softmax")(out)
cnn  = Model(inp, out)

cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

print(cnn.summary())
start_cnn = time.time()
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
                   callbacks=[EarlyStopping(monitor="val_loss", patience=8, mode="min", restore_best_weights=True),
                              ModelCheckpoint("best_cnn.keras", monitor="val_loss", mode="min", save_best_only=True)],
                   verbose=1)
end_cnn = time.time()
print(f"[TIME] CNN training took {(end_cnn - start_cnn)/60:.2f} minutes")

# -----------------------
# [STEP 4] Extract 'feat' + (optional) PCA + standardize (fit on TRAIN only)
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb, yb in ds:
        f = backbone(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_tr_raw, _ = extract_features(make_dataset(idx_tr, shuffle=False, batch_size=BATCH_SIZE))
X_va_raw, _ = extract_features(make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE))
X_te_raw, _ = extract_features(make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE))

if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True, svd_solver="auto")
    pca.fit(X_tr_raw)
    X_tr = pca.transform(X_tr_raw).astype(np.float32)
    X_va = pca.transform(X_va_raw).astype(np.float32)
    X_te = pca.transform(X_te_raw).astype(np.float32)
else:
    X_tr, X_va, X_te = X_tr_raw, X_va_raw, X_te_raw

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr).astype(np.float32)
X_va_s = scaler.transform(X_va).astype(np.float32)
X_te_s = scaler.transform(X_te).astype(np.float32)

# Reassemble to all-node order (transductive)
F = X_tr_s.shape[1]
X_std = np.zeros((len(paths), F), dtype=np.float32)
X_std[idx_tr] = X_tr_s
X_std[idx_va] = X_va_s
X_std[idx_te] = X_te_s

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)
N = X_std.shape[0]
print(f"[STEP 4] Features ready: N={N}, F={F}")

# -----------------------
# [STEP 5] Labels & (optional) class balancing — TRAIN ONLY
# -----------------------
Y_all = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)
Y_train_only = np.zeros_like(Y_all)
Y_train_only[mask_tr] = Y_all[mask_tr]   # zero outside train (safety)

sample_w_train = None
if USE_CLASS_BALANCING:
    counts_tr = np.bincount(labels_int[mask_tr], minlength=num_classes).astype(np.float32)
    class_w = counts_tr.sum() / np.maximum(counts_tr, 1.0)
    class_w = class_w / class_w.mean()
    sample_w_train = np.zeros(len(paths), dtype=np.float32)
    sample_w_train[mask_tr] = class_w[labels_int[mask_tr]]

# -----------------------
# [STEP 6] Graph builders
# -----------------------
from scipy.sparse import coo_matrix, csr_matrix

def build_graph_knn_cosine(X, k=12) -> csr_matrix:
    N_local = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    dist, knn_idx = nbrs.kneighbors(X, return_distance=True)
    rows, cols, data = [], [], []
    for i in range(N_local):
        for j, d in zip(knn_idx[i], dist[i]):
            if i == j: continue
            sim = 1.0 - float(d)
            if sim <= 0: continue
            rows.append(i); cols.append(j); data.append(sim)
    A_dir = coo_matrix((data, (rows, cols)), shape=(N_local, N_local), dtype=np.float32)
    A = A_dir.minimum(A_dir.T).tolil()   # mutual
    A.setdiag(1.0)
    return A.tocsr()

def build_graph_epsilon_cosine(X, eps=0.20) -> csr_matrix:
    radius = max(1e-6, 1.0 - float(eps))  # cosine_distance = 1 - sim
    N_local = X.shape[0]
    nn = NearestNeighbors(metric="cosine", radius=radius).fit(X)
    ind_arrays = nn.radius_neighbors(X, return_distance=False)
    rows, cols, data = [], [], []
    for i, nbrs in enumerate(ind_arrays):
        for j in nbrs:
            if i == j: continue
            rows.append(i); cols.append(j); data.append(1.0)  # unweighted ε-graph
    A_dir = coo_matrix((data, (rows, cols)), shape=(N_local, N_local), dtype=np.float32)
    A = A_dir.minimum(A_dir.T).tolil()   # mutual
    A.setdiag(1.0)
    return A.tocsr()

def build_graph_rbf_kernel(X, gamma=1.0, use_knn=True, k=12) -> csr_matrix:
    N_local = X.shape[0]
    if use_knn:
        nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(X)
        dist, knn_idx = nn.kneighbors(X, return_distance=True)
        rows, cols, data = [], [], []
        for i in range(N_local):
            for j, d in zip(knn_idx[i], dist[i]):
                if i == j: continue
                w = np.exp(-gamma * float(d) * float(d))
                if w <= 0: continue
                rows.append(i); cols.append(j); data.append(w)
        A_dir = coo_matrix((data, (rows, cols)), shape=(N_local, N_local), dtype=np.float32)
        A = A_dir.maximum(A_dir.T).tolil()  # symmetric via max for kernels
        A.setdiag(1.0)
        return A.tocsr()
    else:
        from sklearn.metrics.pairwise import euclidean_distances
        D = euclidean_distances(X, X, squared=True).astype(np.float32)
        W = np.exp(-gamma * D)
        np.fill_diagonal(W, 1.0)
        return csr_matrix(W)

def _infer_group_id(p: str) -> str:
    name = os.path.splitext(os.path.basename(p))[0]
    if "_" in name:
        cand = name.split("_")[0]
        if len(cand) >= 3: return cand
    m = re.search(r"(\d{5,})", name)
    if m: return m.group(1)
    return name

def build_graph_domain_rules(paths_list) -> csr_matrix:
    N_local = len(paths_list)
    groups = {}
    for i, p in enumerate(paths_list):
        gid = _infer_group_id(p)
        groups.setdefault(gid, []).append(i)
    rows, cols, data = [], [], []
    for _, idxs in groups.items():
        if len(idxs) <= 1: continue
        for a in idxs:
            for b in idxs:
                if a == b: continue
                rows.append(a); cols.append(b); data.append(1.0)
    A_dir = coo_matrix((data, (rows, cols)), shape=(N_local, N_local), dtype=np.float32)
    A = A_dir.maximum(A_dir.T).tolil()
    A.setdiag(1.0)
    return A.tocsr()

def make_A_norm(method: str, **kwargs) -> csr_matrix:
    m = method.lower()
    if m == "knn":
        A_local = build_graph_knn_cosine(X_std, k=kwargs.get("k", K_DEFAULT))
    elif m == "epsilon":
        A_local = build_graph_epsilon_cosine(X_std, eps=kwargs.get("eps", 0.20))
    elif m == "rbf":
        A_local = build_graph_rbf_kernel(X_std,
                                         gamma=kwargs.get("gamma", 1.0/max(1, F)),
                                         use_knn=kwargs.get("use_knn", True),
                                         k=kwargs.get("k", K_DEFAULT))
    elif m == "domain":
        A_local = build_graph_domain_rules(paths)
    else:
        raise ValueError(f"Unknown graph method: {method}")
    return gcn_filter(A_local)


# -----------------------
# [STEP X] Compare graph properties across methods
# -----------------------
def compute_graph_properties(A, name="Graph"):
    from scipy.sparse.csgraph import connected_components
    N = A.shape[0]
    nnz_total = A.nnz
    undirected_edges = (nnz_total - N) // 2  # exclude self-loops
    degrees = np.asarray(A.sum(axis=1)).ravel() - 1.0  # minus self-loop
    avg_degree = float(degrees.mean())
    min_degree = int(degrees.min()) if degrees.size else 0
    max_degree = int(degrees.max()) if degrees.size else 0
    possible_edges = N * (N - 1) / 2
    density = undirected_edges / possible_edges if possible_edges > 0 else 0.0
    n_comp, labels_comp = connected_components(A, directed=False)
    largest_cc = int(np.bincount(labels_comp).max()) if labels_comp.size else 0
    return {
        "Graph": name,
        "Nodes": N,
        "Edges": undirected_edges,
        "AvgDeg": round(avg_degree, 2),
        "MinDeg": min_degree,
        "MaxDeg": max_degree,
        "Density": round(density, 6),
        "Components": int(n_comp),
        "LargestCC": largest_cc,
    }

# Build each graph (unnormalized, with self-loops)
A_knn_raw = build_graph_knn_cosine(X_std, k=K_DEFAULT)
A_eps_raw = build_graph_epsilon_cosine(X_std, eps=0.20)
A_rbf_raw = build_graph_rbf_kernel(X_std, gamma=1.0/F, use_knn=True, k=K_DEFAULT)
A_dom_raw = build_graph_domain_rules(paths)

# Collect stats
graph_stats = []
for A, name in [(A_knn_raw,"KNN"),
                (A_eps_raw,"ε-graph"),
                (A_rbf_raw,"RBF"),
                (A_dom_raw,"Domain")]:
    graph_stats.append(compute_graph_properties(A, name=name))

# Convert to DataFrame for pretty table
df_stats = pd.DataFrame(graph_stats,
                        columns=["Graph","Nodes","Edges","AvgDeg","MinDeg","MaxDeg",
                                 "Density","Components","LargestCC"])
print("\n=== Graph Properties Comparison ===")
print(df_stats.to_string(index=False))


# -----------------------
# [STEP 7] GCN builder
# -----------------------
def build_gcn_model(F_dim, N_nodes):
    X_in = Input(shape=(F_dim,), name="X_in")
    A_in = Input((N_nodes,), sparse=True, name="A_in")
    h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="gcn_1")([X_in, A_in])
    h1 = layers.BatchNormalization(name="bn_1")(h1)
    h1 = layers.Activation("relu", name="relu_1")(h1)
    h1 = layers.Dropout(DROPOUT, name="drop_1")(h1)
    res = layers.Dense(HIDDEN, use_bias=False, name="res_proj")(X_in)
    h1 = layers.Add(name="res_add")([h1, res])
    logits = GCNConv(num_classes, activation=None, name="gcn_out")([h1, A_in])
    out    = layers.Activation("softmax", name="softmax")(logits)
    model = Model(inputs=[X_in, A_in], outputs=out, name="gcn_classifier")
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                  weighted_metrics=["accuracy"])
    return model

# -----------------------
# [STEP 8] Tiny grid search per method
# -----------------------
def train_eval_for_graph(method: str, ckpt_tag: str, **a_kwargs):
    A_norm_local = make_A_norm(method, **a_kwargs)
    model = build_gcn_model(F, N)
    t0 = time.time()
    _ = model.fit(
        x=[X_std, A_norm_local],
        y=Y_train_only,
        sample_weight=sample_w_train,
        batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=0,
        validation_data=([X_std, A_norm_local], Y_all, mask_va.astype(np.float32)),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True),
            ModelCheckpoint(f"best_gcn_{ckpt_tag}.keras", monitor="val_loss", mode="min", save_best_only=True),
        ]
    )
    train_minutes = (time.time() - t0) / 60.0
    val_loss, val_acc = model.evaluate([X_std, A_norm_local], Y_all,
                                       sample_weight=mask_va.astype(np.float32),
                                       batch_size=N, verbose=0)
    test_loss, test_acc = model.evaluate([X_std, A_norm_local], Y_all,
                                         sample_weight=mask_te.astype(np.float32),
                                         batch_size=N, verbose=0)
    return train_minutes, val_acc, test_acc

def grid_search_method(method: str):
    trials = []
    if method == "knn":
        for k in [8, 12, 16]:
            tag = f"{method}_k{k}"
            try:
                tmin, va, ta = train_eval_for_graph(method, tag, k=k)
                trials.append(({"k": k}, tmin, va, ta, tag))
            except Exception as e:
                print(f"[GS][{tag}] failed: {e}")
    elif method == "epsilon":
        for eps in [0.10, 0.20, 0.30]:
            tag = f"{method}_eps{eps}"
            try:
                tmin, va, ta = train_eval_for_graph(method, tag, eps=eps)
                trials.append(({"eps": eps}, tmin, va, ta, tag))
            except Exception as e:
                print(f"[GS][{tag}] failed: {e}")
    elif method == "rbf":
        gamma_base = 1.0 / max(1, F)
        for gamma in [gamma_base, 0.5 * gamma_base, 2.0 * gamma_base]:
            tag = f"{method}_gamma{gamma:.4g}"
            try:
                tmin, va, ta = train_eval_for_graph(method, tag, gamma=gamma, use_knn=True, k=K_DEFAULT)
                trials.append(({"gamma": gamma, "use_knn": True, "k": K_DEFAULT}, tmin, va, ta, tag))
            except Exception as e:
                print(f"[GS][{tag}] failed: {e}")
    elif method == "domain":
        tag = f"{method}"
        try:
            tmin, va, ta = train_eval_for_graph(method, tag)
            trials.append(({}, tmin, va, ta, tag))
        except Exception as e:
            print(f"[GS][{tag}] failed: {e}")
    else:
        raise ValueError(method)

    if len(trials) == 0:
        return None

    # pick best by validation accuracy
    best = max(trials, key=lambda x: x[2])
    best_params, tmin, va, ta, tag = best
    return {
        "Graph": method,
        "Params": best_params,
        "Train time (min)": round(tmin, 2),
        "Val Acc": float(va),
        "Test Acc": float(ta),
        "Tag": tag
    }

# Run grid search for all methods
methods = ["knn", "epsilon", "rbf", "domain"]
results = []
for m in methods:
    print(f"\n[GRID] Searching hyperparams for: {m}")
    res = grid_search_method(m)
    if res is None:
        print(f"[GRID] No successful trial for {m}.")
        results.append({"Graph": m, "Params": None, "Train time (min)": None, "Val Acc": None, "Test Acc": None, "Tag": None})
    else:
        print(f"[GRID] Best {m}: params={res['Params']}  val_acc={res['Val Acc']:.4f}  test_acc={res['Test Acc']:.4f}")
        results.append(res)

df_results = pd.DataFrame(results, columns=["Graph", "Params", "Train time (min)", "Val Acc", "Test Acc", "Tag"])
print("\n=== Best per Graph Method ===")
print(df_results[["Graph","Params","Train time (min)","Val Acc","Test Acc"]].to_string(index=False))

# -----------------------
# [STEP 9] Pick overall best method & plot CM + ROC

# -----------------------
# [STEP 9] Evaluate and plot for ALL methods (display only, no saving)
# -----------------------
def build_A_norm_from_params(method: str, params: dict):
    if method == "knn":
        return make_A_norm("knn", k=params["k"])
    elif method == "epsilon":
        return make_A_norm("epsilon", eps=params["eps"])
    elif method == "rbf":
        return make_A_norm("rbf", gamma=params["gamma"], use_knn=True, k=params["k"])
    elif method == "domain":
        return make_A_norm("domain")
    else:
        raise ValueError(method)

def plot_confusion(cm, split_name, method_tag, class_names):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"Confusion Matrix ({split_name}) — {method_tag}")
    plt.colorbar()
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
    plt.tight_layout()
    plt.show()

def plot_roc_curves(mask, split_name, method_tag, y_prob):
    y_true = labels_int[mask]
    y_bin  = label_binarize(y_true, classes=np.arange(num_classes))
    y_pred_bin = y_prob[mask]
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_pred_bin[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_pred_bin.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    plt.figure(figsize=(8, 6))
    for i in range(num_classes):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
    plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves ({split_name}) — {method_tag}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

valid_rows = [r for r in results if r["Val Acc"] is not None and r["Tag"] is not None]
if len(valid_rows) == 0:
    raise RuntimeError("All grid searches failed; cannot proceed.")

# Display metrics and plots for each method
for row in valid_rows:
    method = row["Graph"]
    params = row["Params"] or {}
    tag    = row["Tag"]

    print(f"\n===== [{method.upper()}] Params {params} (tag={tag}) =====")
    A_norm_cur = build_A_norm_from_params(method, params)
    model_path = f"best_gcn_{tag}.keras"
    model_cur  = tf.keras.models.load_model(model_path, custom_objects={"GCNConv": GCNConv})

    # Validation metrics
    val_loss, val_acc = model_cur.evaluate(
        [X_std, A_norm_cur], Y_all,
        sample_weight=mask_va.astype(np.float32),
        batch_size=N, verbose=0
    )
    print(f"[{method}] Validation: loss={val_loss:.4f}  acc={val_acc:.4f}")

    # Test metrics
    test_loss, test_acc = model_cur.evaluate(
        [X_std, A_norm_cur], Y_all,
        sample_weight=mask_te.astype(np.float32),
        batch_size=N, verbose=0
    )
    print(f"[{method}] Test:       loss={test_loss:.4f}  acc={test_acc:.4f}")

    # Predictions for plots
    y_prob_all = model_cur.predict([X_std, A_norm_cur], batch_size=N, verbose=0)
    y_pred_all = np.argmax(y_prob_all, axis=1)

    # Confusion matrices
    cm_val = confusion_matrix(labels_int[mask_va], y_pred_all[mask_va])
    cm_te  = confusion_matrix(labels_int[mask_te], y_pred_all[mask_te])
    plot_confusion(cm_val, "Validation", tag, class_names)
    plot_confusion(cm_te,  "Test",       tag, class_names)

    # ROC curves
    plot_roc_curves(mask_va, "Validation", tag, y_prob_all)
    plot_roc_curves(mask_te,  "Test",       tag, y_prob_all)

print("\nAll methods evaluated and displayed.")


In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()



***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> Hybrid kNN

In [ ]:
# =======================================
# CNN (up to 'feat') -> mutual-kNN GCN
# + graph stats, paper visuals, split pies,
# + one image→pixel-graph per class (32×32)
# =======================================

import os, glob, time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# [STEP 0] Paths & params
# -----------------------
data_dir   = r"D:/Customised_CNN/dataset/Brain_Tumor/four_class"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# CNN training params
LR_CNN     = 1e-3
EPOCHS_CNN = 25
AUGMENT    = True

# Feature post-processing
USE_PCA    = True
PCA_DIM    = 512   # try 256/512

# Graph/GCN params
K          = 12
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# [STEP 1] Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p); labels_int.append(class_to_idx[c])

paths      = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes: {class_names}")

# Stratified splits -> masks
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
mask_te = np.zeros(len(paths), dtype=bool); mask_te[idx_te] = True

# For split pies later
train_labels = labels_int[idx_tr]
val_labels   = labels_int[idx_va]
test_labels  = labels_int[idx_te]

# -----------------------
# [STEP 2] TF dataset
# -----------------------
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)     # [0,1]
    img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)

def augment(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img, seed=SEED)
        img = tf.image.random_flip_up_down(img,   seed=SEED)
        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32, seed=SEED)
        img = tf.image.rot90(img, k)
    return img, label

def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training and AUGMENT:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE, training=True)
val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE, training=False)
test_ds  = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE, training=False)
all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE, training=False)

# -----------------------
# [STEP 2A] Split pies (dataset composition)
# -----------------------
def plot_split_pies():
    CLASS_TYPES = class_names
    N_TYPES = len(CLASS_TYPES)

    def counts_per(labels_vec):
        cnt = np.bincount(labels_vec, minlength=N_TYPES)
        return cnt

    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(20, 14))
    ax = ax.flatten()

    # Train/Val sizes
    ax[0].set_title('Train–Validation Split', fontsize=22)
    ax[0].pie(
        [len(train_labels), len(val_labels)],
        labels=['Train','Validation'],
        colors=['darkcyan', 'orange'],
        autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p * (len(train_labels) + len(val_labels)) / 100),
        explode=(0.1, 0), startangle=85, textprops={'fontsize': 20}
    )

    # Train per-class
    train_cnt = counts_per(train_labels)
    print('Training Counts per class:', dict(zip(CLASS_TYPES, train_cnt)))
    ax[1].set_title('Training Data (per class)', fontsize=22)
    ax[1].pie(
        train_cnt,
        labels=[c.title() for c in CLASS_TYPES],
        colors=['#FAC500','#0BFA00', '#0066FA','#FA0000'][:N_TYPES],
        autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p * train_cnt.sum() / 100),
        explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize': 20}
    )

    # Val per-class
    val_cnt = counts_per(val_labels)
    print('Validation Counts per class:', dict(zip(CLASS_TYPES, val_cnt)))
    ax[2].set_title('Validation Data (per class)', fontsize=22)
    ax[2].pie(
        val_cnt, labels=[c.title() for c in CLASS_TYPES],
        colors=['#FAC500','#0BFA00', '#0066FA','#FA0000'][:N_TYPES],
        autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p * val_cnt.sum() / 100),
        explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize': 20}
    )

    # Test per-class
    test_cnt = counts_per(test_labels)
    print('Testing Counts per class:', dict(zip(CLASS_TYPES, test_cnt)))
    ax[3].set_title('Testing Data (per class)', fontsize=22)
    ax[3].pie(
        test_cnt, labels=[c.title() for c in CLASS_TYPES],
        colors=['#FAC500','#0BFA00', '#0066FA','#FA0000'][:N_TYPES],
        autopct=lambda p: '{:.2f}%\n{:,.0f}'.format(p, p * test_cnt.sum() / 100),
        explode=tuple(0.01 for _ in range(N_TYPES)), textprops={'fontsize': 20}
    )

    plt.tight_layout()
    plt.savefig("fig_split_pies.png", dpi=300)
    plt.show()

plot_split_pies()

# -----------------------
# [STEP 3] CNN backbone that ENDS at 'feat'
# -----------------------
def build_cnn_backbone(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), feat_dim=512):
    return models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat"),  # <-- stops here
    ])

backbone = build_cnn_backbone()

# Temporary head for supervised CNN training
inp  = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
feat = backbone(inp)
out  = layers.Dropout(0.5)(feat)
out  = layers.Dense(num_classes, activation="softmax")(out)
cnn  = Model(inp, out)

cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

print(cnn.summary())
start_cnn = time.time()
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
                   callbacks=[EarlyStopping(monitor="val_loss", patience=8, mode="min", restore_best_weights=True),
                              ModelCheckpoint("best_cnn.keras", monitor="val_loss", mode="min", save_best_only=True)],
                   verbose=1)
end_cnn = time.time()
print(f"[TIME] CNN training took {(end_cnn - start_cnn)/60:.2f} minutes")

# -----------------------
# [STEP 4] Extract 'feat' + (optional) PCA + standardize
# -----------------------
def extract_features(ds):
    feats, ys = [], []
    for xb, yb in ds:
        f = backbone(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_cnn, y_ordered = extract_features(all_ds)
assert np.all(y_ordered == labels_int), "Label order mismatch after feature extraction!"

if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True, svd_solver="auto")
    X = pca.fit_transform(X_cnn).astype(np.float32)
else:
    X = X_cnn

scaler = StandardScaler()
X_std = scaler.fit_transform(X).astype(np.float32)

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)
N, F = X_std.shape
print(f"[STEP 4] Features ready: N={N}, F={F} (backbone feat={X_cnn.shape[1]}; PCA={USE_PCA}, PCA_DIM={PCA_DIM})")

# [STATS A]
print(f"[STATS] Nodes (N) = {N}")
print(f"[STATS] Feature dimension (F) = {F}")

# -----------------------
# [STEP 5] Build mutual-kNN cosine graph
# -----------------------
nbrs = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(X_std)
dist, knn_idx = nbrs.kneighbors(X_std, return_distance=True)

rows, cols, data = [], [], []
for i in range(N):
    for j, d in zip(knn_idx[i], dist[i]):
        if i == j: 
            continue
        sim = 1.0 - float(d)
        if sim <= 0: continue
        rows.append(i); cols.append(j); data.append(sim)

A_dir = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
A_mut = A_dir.minimum(A_dir.T)  # reciprocal edges

# add self-loops
A_mut = A_mut.tolil()
A_mut.setdiag(1.0)
A_mut = A_mut.tocsr()

A_norm = gcn_filter(A_mut)

# [STATS B] edges & graph properties
nnz_total = A_mut.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2

degrees = np.asarray(A_mut.sum(axis=1)).ravel() - 1.0   # minus self-loop
avg_degree = degrees.mean()
max_degree = degrees.max()
min_degree = degrees.min()

possible_edges = N * (N - 1) / 2
density = undirected_edges / possible_edges if possible_edges > 0 else 0.0

from scipy.sparse.csgraph import connected_components
n_comp, labels_comp = connected_components(A_mut, directed=False)
largest_cc = np.bincount(labels_comp).max()

print(f"[STATS] Undirected edges (no self-loops) = {undirected_edges}")
print(f"[STATS] Avg degree (no self-loops) = {avg_degree:.2f} (min={min_degree:.0f}, max={max_degree:.0f})")
print(f"[STATS] Graph density = {density:.6f}")
print(f"[STATS] Connected components = {n_comp}; largest CC size = {largest_cc}")

# -----------------------
# [STEP 5A] Paper visuals
# -----------------------
# t-SNE (2D) — may take time; sample if very large
from sklearn.manifold import TSNE
print("[VIS] Running t-SNE (2D) on features...")
if N > 4000:
    # sample for speed
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(N, size=4000, replace=False)
    X_ts = X_std[sample_idx]
    y_ts = labels_int[sample_idx]
else:
    X_ts, y_ts = X_std, labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = (y_ts == i)
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=6, alpha=0.7, label=cname)
plt.title("t-SNE of Feature Embeddings")
plt.xlabel("t-SNE-1"); plt.ylabel("t-SNE-2")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.savefig("fig_tsne_features.png", dpi=300)
plt.show()

# Degree histogram
plt.figure(figsize=(7, 4))
plt.hist(degrees, bins=range(int(degrees.min()), int(degrees.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node degree (excluding self-loop)")
plt.ylabel("Count")
plt.title("Degree Distribution of Mutual-kNN Graph")
plt.tight_layout()
plt.savefig("fig_degree_hist.png", dpi=300)
plt.show()

# Adjacency snapshot (top-left block)
subN = min(150, N)
A_small = A_mut[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest")
plt.title(f"Adjacency (top-left {subN}×{subN})")
plt.xlabel("Node index"); plt.ylabel("Node index")
plt.tight_layout()
plt.savefig("fig_adjacency_snapshot.png", dpi=300)
plt.show()

# -----------------------
# [STEP 5B] OPTIONAL — one per-class image→pixel-graph (32×32)
# -----------------------
try:
    import cv2, networkx as nx
    from spektral.data import Graph as SpGraph
    px_image_size = (32, 32)

    def px_load_images(data_dir, image_size=(32, 32), classes=None):
        if classes is None:
            classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        imgs, lbls = [], []
        for cname in classes:
            cdir = os.path.join(data_dir, cname)
            if not os.path.isdir(cdir): 
                continue
            for f in os.listdir(cdir):
                p = os.path.join(cdir, f)
                img = cv2.imread(p, cv2.IMREAD_COLOR)
                if img is None:
                    continue
                if img.shape[-1] == 4:
                    img = img[:, :, :3]
                img = cv2.resize(img, image_size, interpolation=cv2.INTER_AREA)
                imgs.append(img); lbls.append(cname)
        name_to_id = {n: i for i, n in enumerate(classes)}
        lbls_int = np.array([name_to_id[s] for s in lbls], dtype=np.int32)
        return np.array(imgs), lbls_int, classes

    def image_to_graph(img):
        h, w, c = img.shape
        n = h * w
        x = (img.reshape(n, c).astype(np.float32)) / 255.0
        a = np.zeros((n, n), dtype=np.float32)
        for i in range(h):
            for j in range(w):
                idx = i * w + j
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < h and 0 <= nj < w:
                            a[idx, ni * w + nj] = 1.0
        return SpGraph(x=x, a=a)

    def visualize_image_and_graph(img_bgr, graph, title, savepath=None):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        G = nx.from_numpy_array(graph.a)
        pos = nx.spring_layout(G, seed=42)
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
        axes[0].imshow(img_rgb); axes[0].set_title(f"{title} — 32×32 image"); axes[0].axis("off")
        nx.draw(G, pos, node_size=12, node_color=graph.x[:, 0], cmap="viridis", with_labels=False, ax=axes[1])
        axes[1].set_title("8-neighbor pixel graph"); axes[1].axis("off")
        plt.tight_layout()
        if savepath: plt.savefig(savepath, dpi=300)
        plt.show()

    px_imgs, px_labels, px_classes = px_load_images(data_dir, image_size=px_image_size, classes=class_names)
    selected_idxs = []
    for cls_id, cname in enumerate(px_classes):
        idxs = np.where(px_labels == cls_id)[0]
        if idxs.size == 0:
            print(f"[WARN] No samples for '{cname}'"); 
            continue
        selected_idxs.append(int(idxs[0]))

    for i in selected_idxs:
        g = image_to_graph(px_imgs[i])
        cname = px_classes[px_labels[i]]
        visualize_image_and_graph(px_imgs[i], g, title=f"{cname}", savepath=f"fig_pixelgraph_oneperclass_{cname}.png")

except Exception as e:
    print(f"[STEP 5B] Skipped pixel-graph viz: {e}")

# -----------------------
# [STEP 6] Class balancing for GCN (optional)
# -----------------------
sample_w_train = mask_tr.astype(np.float32)
if USE_CLASS_BALANCING:
    counts = np.bincount(labels_int, minlength=num_classes).astype(np.float32)
    class_w = counts.sum() / np.maximum(counts, 1.0)
    class_w = class_w / class_w.mean()
    sample_w_train = sample_w_train * class_w[labels_int]

# -----------------------
# [STEP 7] Graph Convolutional Network (GCN)
# -----------------------
X_in = Input(shape=(F,), name="X_in")
A_in = Input((N,), sparse=True, name="A_in")

h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="gcn_1")([X_in, A_in])
h1 = layers.BatchNormalization(name="bn_1")(h1)
h1 = layers.Activation("relu", name="relu_1")(h1)
h1 = layers.Dropout(DROPOUT, name="drop_1")(h1)
res = layers.Dense(HIDDEN, use_bias=False, name="res_proj")(X_in)
h1 = layers.Add(name="res_add")([h1, res])

logits = GCNConv(num_classes, activation=None, name="gcn_out")([h1, A_in])
out    = layers.Activation("softmax", name="softmax")(logits)

gcn = Model(inputs=[X_in, A_in], outputs=out, name="gcn_classifier")
gcn.compile(
    optimizer=tf.keras.optimizers.Adam(LR_GCN),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    weighted_metrics=["accuracy"]
)

print(gcn.summary())

start_gcn = time.time()
hist_gcn = gcn.fit(
    x=[X_std, A_norm],
    y=to_categorical(labels_int, num_classes=num_classes),
    sample_weight=sample_w_train,             # only train nodes contribute
    batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1,
    validation_data=([X_std, A_norm], to_categorical(labels_int, num_classes=num_classes), mask_va.astype(np.float32)),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True),
        ModelCheckpoint("best_gcn.keras", monitor="val_loss", mode="min", save_best_only=True),
    ]
)
end_gcn = time.time()
print(f"[TIME] GCN training took {(end_gcn - start_gcn)/60:.2f} minutes")

# -----------------------
# [STEP 8] Validation report
# -----------------------
val_loss, val_acc = gcn.evaluate(
    x=[X_std, A_norm], y=to_categorical(labels_int, num_classes),
    sample_weight=mask_va.astype(np.float32),
    batch_size=N, verbose=0
)
print(f"[GCN] Validation: loss={val_loss:.4f}  acc={val_acc:.4f}")

y_prob_all = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
y_pred_all = np.argmax(y_prob_all, axis=1)
y_true_all = labels_int

print("\n[GCN] Classification Report (VALIDATION):")
print(classification_report(y_true_all[mask_va], y_pred_all[mask_va], target_names=class_names))

cm_val = confusion_matrix(y_true_all[mask_va], y_pred_all[mask_va])
print("[GCN] Confusion Matrix (VALIDATION):\n", cm_val)

plt.figure(figsize=(6, 6))
plt.imshow(cm_val, cmap="Blues")
plt.title("GCN Confusion Matrix (Validation)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm_val[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.savefig("fig_cm_val.png", dpi=300); plt.show()

# -----------------------
# [STEP 9] Test report
# -----------------------
test_loss, test_acc = gcn.evaluate(
    x=[X_std, A_norm], y=to_categorical(labels_int, num_classes),
    sample_weight=mask_te.astype(np.float32),
    batch_size=N, verbose=0
)
print(f"[GCN] Test: loss={test_loss:.4f}  acc={test_acc:.4f}")

print("\n[GCN] Classification Report (TEST):")
print(classification_report(y_true_all[mask_te], y_pred_all[mask_te], target_names=class_names))

cm_test = confusion_matrix(y_true_all[mask_te], y_pred_all[mask_te])
print("[GCN] Confusion Matrix (TEST):\n", cm_test)

plt.figure(figsize=(6, 6))
plt.imshow(cm_test, cmap="Blues")
plt.title("GCN Confusion Matrix (Test)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm_test[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.savefig("fig_cm_test.png", dpi=300); plt.show()

# -----------------------
# [STEP 10] (Optional) ROC curves
# -----------------------
def plot_roc(mask, name, y_prob):
    y_true = labels_int[mask]
    y_bin  = label_binarize(y_true, classes=np.arange(num_classes))
    y_pred_bin = y_prob[mask]
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_pred_bin[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_pred_bin.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    plt.figure(figsize=(8, 6))
    for i in range(num_classes):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
    plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"GCN ROC Curves ({name})"); plt.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(f"fig_roc_{name.lower()}.png", dpi=300); plt.show()

plot_roc(mask_va, "Validation", y_prob_all)
plot_roc(mask_te, "Test",       y_prob_all)


In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
# === Ensemble‑only Results: CNN + GCN (weighted late fusion) ===
# Keeps full training pipeline but ONLY prints/plots ensemble metrics.
# Works with: TensorFlow 2.x, Spektral 1.x/2.x, scikit-learn, numpy, matplotlib

import os, glob
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# 0) Paths & params
# -----------------------
data_dir   = r"D:/Customised_CNN/dataset/Brain_Tumor/four_class"   # <-- your dataset
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# CNN training params
LR_CNN     = 1e-3
EPOCHS_CNN = 50
AUGMENT    = True  # turn off if you want pure training without augmentation

# Feature post-processing
USE_PCA    = True
PCA_DIM    = 512   # try 256/512

# Graph/GCN params
K          = 12        # mutual kNN (tune 8/12/16/20)
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 200
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05
VAL_SPLIT  = 0.2
TEST_SIZE  = 0.2

# Class balancing for GCN (optional)
USE_CLASS_BALANCING = True

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------
# 1) Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths      = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes: {class_names}")

# Stratified splits -> masks
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
mask_te = np.zeros(len(paths), dtype=bool); mask_te[idx_te] = True

# -----------------------
# 2) TF dataset (pure TF ops)
# -----------------------
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)     # [0,1]
    img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)

def augment(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img, seed=SEED)
        img = tf.image.random_flip_up_down(img,   seed=SEED)
        img = tfa_random_rotate(img)
    return img, label

def tfa_random_rotate(x):
    # simple 0/90/180/270 rotation without tf-addons
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32, seed=SEED)
    return tf.image.rot90(x, k)

def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training and AUGMENT:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE, training=True)
val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE, training=False)
test_ds  = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE, training=False)
all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE, training=False)

# -----------------------
# 3) CNN (as feature extractor)
# -----------------------
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.Flatten(),
        layers.Dense(512, activation="relu", name="feat"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

cnn = create_model()
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

ckpt_cnn = "best_cnn.keras"
callbacks_cnn = [
    EarlyStopping(monitor="val_loss", patience=8, mode="min", restore_best_weights=True),
    ModelCheckpoint(ckpt_cnn, monitor="val_loss", mode="min", save_best_only=True),
]
print(cnn.summary())
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN, callbacks=callbacks_cnn, verbose=1)

# -----------------------
# 4) Extract features for ALL images from 'feat' layer
# -----------------------
feat_extractor = Model(inputs=cnn.input, outputs=cnn.get_layer("feat").output)

def extract_features(ds):
    feats, ys = [], []
    for xb, yb in ds:
        f = feat_extractor(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

X_cnn, y_ordered = extract_features(all_ds)
assert np.all(y_ordered == labels_int), "Label order mismatch after feature extraction!"

# CNN probabilities for ALL images (same order as labels_int / all_ds)
y_prob_cnn_all = cnn.predict(all_ds, verbose=0)
assert y_prob_cnn_all.shape == (len(labels_int), num_classes)

# Optional PCA (often helps)
if USE_PCA:
    pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True, svd_solver="auto")
    X = pca.fit_transform(X_cnn).astype(np.float32)
else:
    X = X_cnn

# Standardize for cosine metric
scaler = StandardScaler()
X_std = scaler.fit_transform(X).astype(np.float32)

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)
N, F = X_std.shape
print(f"Features ready: N={N}, F={F} (from CNN feat=512; PCA={USE_PCA}, PCA_DIM={PCA_DIM})")

# -----------------------
# 5) Build mutual kNN cosine graph
# -----------------------
nbrs = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(X_std)
dist, knn_idx = nbrs.kneighbors(X_std, return_distance=True)

rows, cols, data = [], [], []
for i in range(N):
    for j, d in zip(knn_idx[i], dist[i]):
        if i == j:
            continue
        sim = 1.0 - float(d)   # cosine similarity in [0,1]
        if sim <= 0:
            continue
        rows.append(i); cols.append(j); data.append(sim)

A_dir = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
A_mut = A_dir.minimum(A_dir.T)     # mutual (reciprocal) edges

# self-loops with weight 1
A_mut = A_mut.tolil()
A_mut.setdiag(1.0)
A_mut = A_mut.tocsr()

A_norm = gcn_filter(A_mut)

# -----------------------
# 6) Optional class balancing for GCN
# -----------------------
sample_w_train = mask_tr.astype(np.float32)
if USE_CLASS_BALANCING:
    counts = np.bincount(labels_int, minlength=num_classes).astype(np.float32)
    class_w = counts.sum() / np.maximum(counts, 1.0)
    class_w = class_w / class_w.mean()
    sample_w_train = sample_w_train * class_w[labels_int]

# -----------------------
# 7) GCN model (BN + residual), label smoothing
# -----------------------
from tensorflow.keras.layers import BatchNormalization

X_in = Input(shape=(F,), name="X_in")
A_in = Input((N,), sparse=True, name="A_in")

h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD))([X_in, A_in])
h1 = BatchNormalization()(h1)
h1 = tf.nn.relu(h1)
h1 = layers.Dropout(DROPOUT)(h1)

res = layers.Dense(HIDDEN, use_bias=False)(X_in)  # residual projection
h1 = layers.Add()([h1, res])

out = GCNConv(num_classes, activation="softmax")([h1, A_in])

gcn = Model(inputs=[X_in, A_in], outputs=out)
gcn.compile(
    optimizer=tf.keras.optimizers.Adam(LR_GCN),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    weighted_metrics=["accuracy"]
)

ckpt_gcn = "best_gcn.keras"
callbacks_gcn = [
    EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True),
    ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True),
]

hist_gcn = gcn.fit(
    x=[X_std, A_norm],
    y=y_onehot,
    sample_weight=sample_w_train,                     # only train nodes contribute
    batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1,
    validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)),
    callbacks=callbacks_gcn
)

# -----------------------
# Ensemble ONLY (validation + test)
# -----------------------
# Get probabilities for ALL nodes from GCN

y_prob_all = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
y_true_all = labels_int

# Grid-search alpha on VALIDATION
alphas = np.linspace(0.0, 1.0, 21)  # 0.00, 0.05, ..., 1.00
best_alpha, best_val_acc = None, -1.0

y_true_val = y_true_all[mask_va]

for a in alphas:
    y_ens_val = a * y_prob_cnn_all[mask_va] + (1.0 - a) * y_prob_all[mask_va]
    y_pred_ens_val = np.argmax(y_ens_val, axis=1)
    acc = accuracy_score(y_true_val, y_pred_ens_val)
    if acc > best_val_acc:
        best_val_acc = acc
        best_alpha = float(a)

print(f"[Ensemble] Best alpha on VALIDATION = {best_alpha:.2f}  (val acc={best_val_acc:.4f})")

# Detailed VALIDATION metrics for ensemble

y_ens_val = best_alpha * y_prob_cnn_all[mask_va] + (1.0 - best_alpha) * y_prob_all[mask_va]
y_pred_ens_val = np.argmax(y_ens_val, axis=1)

print("\n[Ensemble] Classification Report (VALIDATION):")
print(classification_report(y_true_val, y_pred_ens_val, target_names=class_names))

cm_val_ens = confusion_matrix(y_true_val, y_pred_ens_val)
print("[Ensemble] Confusion Matrix (VALIDATION):\n", cm_val_ens)

plt.figure(figsize=(6, 6))
plt.imshow(cm_val_ens, cmap="Blues")
plt.title("Ensemble Confusion Matrix (Validation)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm_val_ens[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.show()

# ROC (validation) for ensemble

y_val_binE  = label_binarize(y_true_val, classes=np.arange(num_classes))
fprE, tprE, roc_aucE = {}, {}, {}
for i in range(num_classes):
    fprE[i], tprE[i], _ = roc_curve(y_val_binE[:, i], y_ens_val[:, i])
    roc_aucE[i] = auc(fprE[i], tprE[i])
fprE["micro"], tprE["micro"], _ = roc_curve(y_val_binE.ravel(), y_ens_val.ravel())
roc_aucE["micro"] = auc(fprE["micro"], tprE["micro"])

plt.figure(figsize=(8, 6))
for i in range(num_classes):
    plt.plot(fprE[i], tprE[i], label=f"{class_names[i]} (AUC={roc_aucE[i]:.2f})")
plt.plot(fprE["micro"], tprE["micro"], linestyle="--", label=f"Micro (AUC={roc_aucE['micro']:.2f})")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Ensemble ROC Curves (Validation)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

# TEST metrics for ensemble

y_true_test = y_true_all[mask_te]
y_ens_test  = best_alpha * y_prob_cnn_all[mask_te] + (1.0 - best_alpha) * y_prob_all[mask_te]
y_pred_ens_test = np.argmax(y_ens_test, axis=1)

test_acc_ens = accuracy_score(y_true_test, y_pred_ens_test)
print(f"[Ensemble] Test: acc={test_acc_ens:.4f}  (alpha={best_alpha:.2f})")

print("\n[Ensemble] Classification Report (TEST):")
print(classification_report(y_true_test, y_pred_ens_test, target_names=class_names))

cm_test_ens = confusion_matrix(y_true_test, y_pred_ens_test)
print("[Ensemble] Confusion Matrix (TEST):\n", cm_test_ens)

plt.figure(figsize=(6, 6))
plt.imshow(cm_test_ens, cmap="Blues")
plt.title("Ensemble Confusion Matrix (Test)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm_test_ens[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.show()

# ROC (test) for ensemble

y_test_binE  = label_binarize(y_true_test, classes=np.arange(num_classes))
fprTE, tprTE, roc_aucTE = {}, {}, {}
for i in range(num_classes):
    fprTE[i], tprTE[i], _ = roc_curve(y_test_binE[:, i], y_ens_test[:, i])
    roc_aucTE[i] = auc(fprTE[i], tprTE[i])
fprTE["micro"], tprTE["micro"], _ = roc_curve(y_test_binE.ravel(), y_ens_test.ravel())
roc_aucTE["micro"] = auc(fprTE["micro"], tprTE["micro"]) 

plt.figure(figsize=(8, 6))
for i in range(num_classes):
    plt.plot(fprTE[i], tprTE[i], label=f"{class_names[i]} (AUC={roc_aucTE[i]:.2f})")
plt.plot(fprTE["micro"], tprTE["micro"], linestyle="--", label=f"Micro (AUC={roc_aucTE['micro']:.2f})")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Ensemble ROC Curves (Test)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

# Note: All GCN‑only and CNN‑only reports/curves have been removed to keep output focused on the ensemble.


In [ ]:
"""
K‑Fold CV + Explainability (SHAP, LIME, Grad‑CAM) for CNN+GCN Ensemble — FIXED
-------------------------------------------------------------------------------
- 5‑fold Stratified CV with inner validation split
- Ensemble = weighted late fusion (alpha tuned on val set each fold)
- Saves per‑fold: classification report (txt + csv), confusion matrix (.npy + png),
  ROC curves (per‑class + micro + macro), learning curves (loss & accuracy) for CNN & GCN,
  and a JSON summary with best alpha / accuracies.
- After CV, runs LIME, SHAP, Grad‑CAM on the best fold’s TEST set (up to 5 images/class).

Notes
- No SMOTE (as requested). All related code has been removed.
- GCN is trained transductively with sample‑weight masks for train/val nodes.
- Keep K, PCA_DIM, and epochs modest if you hit memory constraints.

Requirements
- TensorFlow 2.x, scikit‑learn, spektral 1.x/2.x, numpy, matplotlib, pandas
- SHAP (>=0.42), lime (>=0.2), scikit‑image, opencv‑python
"""
import os, glob, json, random
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd

from typing import Dict, List, Tuple

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, accuracy_score
)

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import BatchNormalization

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# Optional (recommended) for LIME/SHAP visuals
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# -----------------------
# 0) Paths & params
# -----------------------
data_dir   = r"D:/Customised_CNN/dataset/Brain_Tumor/four_class"   # <-- set your dataset root
OUT_DIR    = r"./cv_explain_out"
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 123

# Cross‑validation
K_FOLDS    = 5    # k‑folds
VAL_SPLIT  = 0.2  # inner validation split from each training fold chunk

# CNN training params
LR_CNN     = 1e-3
EPOCHS_CNN = 50
AUGMENT    = True  # turn off for pure training

# Feature post‑processing
USE_PCA    = True
PCA_DIM    = 512   # try 256/512

# Graph/GCN params
K          = 12        # mutual kNN (tune 8/12/16/20)
LR_GCN     = 5e-3
WD         = 5e-4
EPOCHS_GCN = 80
HIDDEN     = 64
DROPOUT    = 0.3
LABEL_SMOOTH = 0.05

# Class balancing for GCN (optional)
USE_CLASS_BALANCING = True

# Reproducibility & GPU friendliness
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
try:
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception:
    pass

# -----------------------
# 1) Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths      = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes: {class_names}")

# -----------------------
# 2) TF dataset utils
# -----------------------

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)     # [0,1]
    img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, tf.cast(label, tf.int32)


def tfa_random_rotate(x):
    # simple 0/90/180/270 rotation without tf-addons
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32, seed=SEED)
    return tf.image.rot90(x, k)


def augment(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img, seed=SEED)
        img = tf.image.random_flip_up_down(img,   seed=SEED)
        img = tfa_random_rotate(img)
    return img, label


def make_dataset(indexes, shuffle=True, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training and AUGMENT:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# -----------------------
# 3) CNN (as feature extractor)
# -----------------------

def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="conv1"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu", name="conv2"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu", name="conv3"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu", name="conv4"),
        layers.Flatten(),
        layers.Dense(512, activation="relu", name="feat"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax", name="pred")
    ])
    return model

# -----------------------
# 4) Feature extraction helper
# -----------------------

def extract_features(ds, feat_extractor: Model):
    feats, ys = [], []
    for xb, yb in ds:
        f = feat_extractor(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats).astype(np.float32), np.concatenate(ys)

# -----------------------
# 5) Graph utilities
# -----------------------

def build_graph_features(X_cnn: np.ndarray):
    # Optional PCA
    if USE_PCA:
        pca = PCA(n_components=PCA_DIM, random_state=SEED, whiten=True, svd_solver="auto")
        X = pca.fit_transform(X_cnn).astype(np.float32)
    else:
        X = X_cnn
    # Standardize for cosine metric
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X).astype(np.float32)

    # kNN graph (cosine)
    N, F = X_std.shape
    nbrs = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(X_std)
    dist, knn_idx = nbrs.kneighbors(X_std, return_distance=True)

    rows, cols, data = [], [], []
    for i in range(N):
        for j, d in zip(knn_idx[i], dist[i]):
            if i == j:
                continue
            sim = 1.0 - float(d)   # cosine similarity in [0,1]
            if sim <= 0:
                continue
            rows.append(i); cols.append(j); data.append(sim)

    A_dir = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
    A_mut = A_dir.minimum(A_dir.T)     # mutual (reciprocal) edges

    # self-loops with weight 1
    A_mut = A_mut.tolil()
    A_mut.setdiag(1.0)
    A_mut = A_mut.tocsr()

    A_norm = gcn_filter(A_mut)
    return X_std, A_norm

# -----------------------
# 6) GCN model factory
# -----------------------

def build_gcn(F_in: int, N: int):
    X_in = Input(shape=(F_in,), name="X_in")
    A_in = Input((N,), sparse=True, name="A_in")

    h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD))([X_in, A_in])
    h1 = BatchNormalization()(h1)
    h1 = tf.nn.relu(h1)
    h1 = layers.Dropout(DROPOUT)(h1)

    res = layers.Dense(HIDDEN, use_bias=False)(X_in)  # residual projection
    h1 = layers.Add()([h1, res])

    out = GCNConv(num_classes, activation="softmax")([h1, A_in])

    gcn = Model(inputs=[X_in, A_in], outputs=out)
    gcn.compile(
        optimizer=tf.keras.optimizers.Adam(LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
        weighted_metrics=["accuracy"],
    )
    return gcn

# -----------------------
# 7) Explainability helpers (Grad‑CAM, SHAP, LIME)
# -----------------------

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_outputs[0]
    cam = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], IMG_SIZE)
    return cam.numpy().squeeze()


def gradcam_sets(cnn_model, X_test, y_test, out_dir, last_conv='conv4', num_each=5):
    os.makedirs(out_dir, exist_ok=True)
    preds = cnn_model.predict(X_test, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    mis_idx = np.where(y_pred != y_test)[0][:num_each]
    cor_idx = np.where(y_pred == y_test)[0][:num_each]

    import cv2
    for tag, idxs in [("misclassified", mis_idx), ("correct", cor_idx)]:
        d = os.path.join(out_dir, tag); os.makedirs(d, exist_ok=True)
        for i, idx in enumerate(idxs):
            img = X_test[idx]
            heat = make_gradcam_heatmap(np.expand_dims(img,0), cnn_model, last_conv)
            heatmap = (heat * 255).astype(np.uint8)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
            base = (np.clip(img*255,0,255)).astype(np.uint8)
            overlay = cv2.addWeighted(base, 0.6, heatmap, 0.4, 0)
            cv2.imwrite(os.path.join(d, f"{tag}_{i:03d}_true{class_names[y_test[idx]]}_pred{class_names[y_pred[idx]]}.png"), overlay[:, :, ::-1])


def lime_explain_colored(cnn_model, X_imgs, y_labels, out_dir, num_features=10, num_samples=1000):
    os.makedirs(out_dir, exist_ok=True)
    explainer = lime_image.LimeImageExplainer()

    def predict_fn(x):
        x = tf.convert_to_tensor(x, dtype=tf.float32)
        if tf.reduce_max(x).numpy() > 1.0:
            x = x / 255.0
        x = tf.image.resize(x, IMG_SIZE)
        return cnn_model(x, training=False).numpy()

    for i, (img, y) in enumerate(zip(X_imgs, y_labels)):
        # LIME expects uint8 images typically
        img_uint8 = (np.clip(img * 255.0, 0, 255)).astype(np.uint8)
        exp = explainer.explain_instance(
            image=img_uint8, classifier_fn=predict_fn, top_labels=1,
            hide_color=0, num_samples=num_samples
        )
        top_label = exp.top_labels[0]
        temp, mask = exp.get_image_and_mask(
            label=top_label, positive_only=False, hide_rest=False, num_features=num_features
        )
        # overlays
        fig, axs = plt.subplots(1, 2, figsize=(8, 4))
        axs[0].imshow(img_uint8); axs[0].set_title(f"Input: {class_names[y]}"); axs[0].axis('off')
        axs[1].imshow(mark_boundaries(temp / 255.0, mask)); axs[1].set_title("LIME overlay"); axs[1].axis('off')
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"lime_{i:03d}_true{class_names[y]}_pred{class_names[top_label]}.png"), dpi=160)
        plt.close(fig)


def shap_explain_image(cnn_model, X_imgs, out_dir, class_names):
    os.makedirs(out_dir, exist_ok=True)

    # Try gradient-based Explainer; fallback to masker-based if needed
    background = X_imgs[: min(64, len(X_imgs))]
    try:
        explainer = shap.GradientExplainer((cnn_model.input, cnn_model.output), background)
        shap_values = explainer.shap_values(X_imgs)
        for c in range(len(class_names)):
            try:
                fig = plt.figure(figsize=(6, 4))
                shap.image_plot(shap_values[c], X_imgs, show=False)
                plt.suptitle(f"SHAP – class {class_names[c]}")
                fig.savefig(os.path.join(out_dir, f"shap_grad_{c}_{class_names[c]}.png"), dpi=160)
                plt.close(fig)
            except Exception as e:
                print("[SHAP-grad] Skip class", c, e)
    except Exception as e:
        print("[SHAP] GradientExplainer failed, falling back to masker-based explainer:", e)
        masker = shap.maskers.Image("blur(16,16)", X_imgs[0].shape)
        f = lambda z: cnn_model(tf.image.resize(tf.convert_to_tensor(z, tf.float32)/255.0, IMG_SIZE), training=False).numpy()
        ex = shap.Explainer(f, masker, output_names=class_names)
        # limit to a handful to keep runtime sane
        M = min(8, len(X_imgs))
        vals = ex(X_imgs[:M], max_evals=500, batch_size=32)
        fig = plt.figure(figsize=(8, 6))
        shap.image_plot(vals, show=False)
        fig.savefig(os.path.join(out_dir, f"shap_masker_summary.png"), dpi=160)
        plt.close(fig)


def run_explainability_v2(best_fold_dir: str, best_cnn_path: str, test_indices: np.ndarray):
    print("[Explain] Loading best CNN for explanations…", best_cnn_path)
    cnn_best = create_model(); cnn_best.load_weights(best_cnn_path)
    # Build test set tensors
    ds_test = make_dataset(test_indices, shuffle=False, batch_size=BATCH_SIZE, training=False)
    imgs, ys = [], []
    for xb, yb in ds_test:
        imgs.append(xb.numpy()); ys.append(yb.numpy())
    X_test = np.concatenate(imgs, axis=0)
    y_test = np.concatenate(ys, axis=0)

    # Select up to 5 per class in test
    per_class = {c: [] for c in range(num_classes)}
    for i, y in enumerate(y_test):
        if len(per_class[y]) < 5:
            per_class[y].append(i)
        if all(len(v) == 5 or len(v) == int((y_test==k).sum()) for k, v in per_class.items()):
            pass
    sel_idx = sorted({i for v in per_class.values() for i in v})
    X_sel = X_test[sel_idx]; y_sel = y_test[sel_idx]

    # LIME overlays
    lime_explain_colored(cnn_best, X_sel, y_sel, out_dir=os.path.join(best_fold_dir, "lime"))
    # SHAP (gradient or masker fallback)
    shap_explain_image(cnn_best, X_sel, out_dir=os.path.join(best_fold_dir, "shap"), class_names=class_names)
    # Grad‑CAM on correct/misclassified
    gradcam_sets(cnn_best, X_test, y_test, out_dir=os.path.join(best_fold_dir, "gradcam_sets"), last_conv='conv4', num_each=5)

# -----------------------
# 8) Metrics/plots helpers
# -----------------------

def save_history_plots(history: Dict[str, List[float]], out_dir: str, prefix: str):
    os.makedirs(out_dir, exist_ok=True)
    # Loss
    plt.figure(figsize=(6,4))
    plt.plot(history.get('loss', []), label='train')
    if 'val_loss' in history:
        plt.plot(history['val_loss'], label='val')
    plt.title(f"{prefix} Loss")
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix.lower()}_loss_curve.png"), dpi=180)
    plt.close()
    # Accuracy
    if 'accuracy' in history or 'val_accuracy' in history:
        plt.figure(figsize=(6,4))
        if 'accuracy' in history:
            plt.plot(history['accuracy'], label='train')
        if 'val_accuracy' in history:
            plt.plot(history['val_accuracy'], label='val')
        plt.title(f"{prefix} Accuracy")
        plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix.lower()}_accuracy_curve.png"), dpi=180)
        plt.close()


def plot_roc_multiclass(y_true: np.ndarray, y_score: np.ndarray, class_names: List[str], out_png: str, out_json: str):
    # y_true: (N,), y_score: (N, C)
    y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
    fpr, tpr, roc_auc = {}, {}, {}
    # per class
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    # micro
    fpr['micro'], tpr['micro'], _ = roc_curve(y_bin.ravel(), y_score.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
    # macro
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(class_names))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(class_names)):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(class_names)
    roc_auc['macro'] = auc(all_fpr, mean_tpr)

    # Plot
    plt.figure(figsize=(7,6))
    for i, name in enumerate(class_names):
        plt.plot(fpr[i], tpr[i], lw=1.2, label=f"{name} (AUC={roc_auc[i]:.3f})")
    plt.plot(fpr['micro'], tpr['micro'], lw=2, linestyle='--', label=f"micro (AUC={roc_auc['micro']:.3f})")
    plt.plot([0,1], [0,1], lw=1, color='gray')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('Ensemble ROC – per class / micro')
    plt.legend(loc='lower right', fontsize=8)
    plt.tight_layout(); plt.savefig(out_png, dpi=180); plt.close()

    # Save AUCs
    with open(out_json, 'w') as f:
        json.dump({k: float(v) for k, v in roc_auc.items()}, f, indent=2)

# -----------------------
# 9) CV loop
# -----------------------

def run_cv():
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    all_idx = np.arange(len(paths))

    fold_summaries = []
    best_val = -1.0
    best_fold_artifacts = None

    for fold, (trainval_idx, test_idx) in enumerate(skf.split(all_idx, labels_int), start=1):
        print(f"\n==================== Fold {fold}/{K_FOLDS} ====================")
        # Inner validation split
        idx_tr, idx_va = train_test_split(
            trainval_idx,
            test_size=VAL_SPLIT,
            random_state=SEED,
            stratify=labels_int[trainval_idx],
        )

        # Datasets
        train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE, training=True)
        val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE, training=False)
        test_ds  = make_dataset(test_idx, shuffle=False, batch_size=BATCH_SIZE, training=False)
        all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE, training=False)

        fold_dir = os.path.join(OUT_DIR, f"fold_{fold:02d}")
        os.makedirs(fold_dir, exist_ok=True)

        # ----- CNN -----
        tf.keras.backend.clear_session()
        cnn = create_model()
        cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"]) 
        ckpt_cnn = os.path.join(fold_dir, "best_cnn.keras")
        callbacks_cnn = [
            EarlyStopping(monitor="val_loss", patience=8, mode="min", restore_best_weights=True),
            ModelCheckpoint(ckpt_cnn, monitor="val_loss", mode="min", save_best_only=True),
        ]
        print(cnn.summary())
        hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
                            callbacks=callbacks_cnn, verbose=1)
        # Save curves + history
        save_history_plots(hist_cnn.history, fold_dir, prefix="CNN")
        with open(os.path.join(fold_dir, "cnn_history.json"), 'w') as f:
            json.dump(hist_cnn.history, f)

        # Extract features for ALL images
        feat_extractor = Model(inputs=cnn.input, outputs=cnn.get_layer("feat").output)
        X_cnn, y_ordered = extract_features(all_ds, feat_extractor)
        assert np.all(y_ordered == labels_int), "Label order mismatch after feature extraction!"

        # CNN probabilities for ALL images
        y_prob_cnn_all = cnn.predict(all_ds, verbose=0)
        assert y_prob_cnn_all.shape == (len(labels_int), num_classes)

        # Build graph features & adjacency
        X_std, A_norm = build_graph_features(X_cnn)
        N, F = X_std.shape
        y_onehot = tf.keras.utils.to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

        # Masks
        mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
        mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
        mask_te = np.zeros(len(paths), dtype=bool); mask_te[test_idx] = True

        # Class balancing
        sample_w_train = mask_tr.astype(np.float32)
        if USE_CLASS_BALANCING:
            counts = np.bincount(labels_int, minlength=num_classes).astype(np.float32)
            class_w = counts.sum() / np.maximum(counts, 1.0)
            class_w = class_w / class_w.mean()
            sample_w_train = sample_w_train * class_w[labels_int]

        # ----- GCN -----
        gcn = build_gcn(F, N)
        ckpt_gcn = os.path.join(fold_dir, "best_gcn.keras")
        callbacks_gcn = [
            EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True),
            ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True),
        ]
        hist_gcn = gcn.fit(
            x=[X_std, A_norm], y=y_onehot,
            sample_weight=sample_w_train,
            batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1,
            validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)),
            callbacks=callbacks_gcn,
        )
        save_history_plots(hist_gcn.history, fold_dir, prefix="GCN")
        with open(os.path.join(fold_dir, "gcn_history.json"), 'w') as f:
            json.dump(hist_gcn.history, f)

        # ----- Ensemble eval -----
        y_prob_gcn_all = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
        y_true_all = labels_int

        # Grid‑search alpha on VALIDATION
        alphas = np.linspace(0.0, 1.0, 21)
        y_true_val = y_true_all[mask_va]
        best_alpha, best_val_acc = None, -1.0
        for a in alphas:
            y_ens_val = a * y_prob_cnn_all[mask_va] + (1.0 - a) * y_prob_gcn_all[mask_va]
            y_pred_ens_val = np.argmax(y_ens_val, axis=1)
            acc = accuracy_score(y_true_val, y_pred_ens_val)
            if acc > best_val_acc:
                best_val_acc = acc
                best_alpha = float(a)

        # Test metrics (ensemble)
        y_true_test = y_true_all[mask_te]
        y_ens_test  = best_alpha * y_prob_cnn_all[mask_te] + (1.0 - best_alpha) * y_prob_gcn_all[mask_te]
        y_pred_ens_test = np.argmax(y_ens_test, axis=1)
        test_acc_ens = accuracy_score(y_true_test, y_pred_ens_test)

        # Reports
        rep_text = classification_report(y_true_test, y_pred_ens_test, target_names=class_names)
        rep_dict = classification_report(y_true_test, y_pred_ens_test, target_names=class_names, output_dict=True)
        cm = confusion_matrix(y_true_test, y_pred_ens_test)

        # Save per‑fold artifacts
        with open(os.path.join(fold_dir, "summary.json"), "w") as f:
            json.dump({
                "fold": fold,
                "best_alpha": best_alpha,
                "val_acc": float(best_val_acc),
                "test_acc": float(test_acc_ens),
            }, f, indent=2)
        np.save(os.path.join(fold_dir, "confusion_matrix.npy"), cm)
        with open(os.path.join(fold_dir, "classification_report.txt"), 'w') as f:
            f.write(rep_text)
        pd.DataFrame(rep_dict).T.to_csv(os.path.join(fold_dir, "classification_report.csv"))

        # Confusion matrix plot
        plt.figure(figsize=(6,6))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"Ensemble Confusion Matrix – Fold {fold}")
        plt.colorbar()
        plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
        plt.yticks(range(num_classes), class_names)
        plt.xlabel("Predicted"); plt.ylabel("True")
        for i in range(num_classes):
            for j in range(num_classes):
                plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
        plt.tight_layout(); plt.savefig(os.path.join(fold_dir, "cm.png"), dpi=180)
        plt.close()

        # ROC curves (per class + micro + macro)
        plot_roc_multiclass(
            y_true=y_true_test,
            y_score=y_ens_test,
            class_names=class_names,
            out_png=os.path.join(fold_dir, "roc_curves.png"),
            out_json=os.path.join(fold_dir, "roc_auc.json")
        )

        # Save raw preds for potential later analysis
        np.savez_compressed(
            os.path.join(fold_dir, "test_preds_logits_probs.npz"),
            y_true=y_true_test, y_pred=y_pred_ens_test, y_prob=y_ens_test
        )

        fold_summaries.append({
            "fold": fold,
            "val_acc": float(best_val_acc),
            "test_acc": float(test_acc_ens),
            "alpha": float(best_alpha),
            "cnn_ckpt": ckpt_cnn,
            "gcn_ckpt": ckpt_gcn,
            "test_indices": test_idx.tolist(),
            "dir": fold_dir,
        })

        if best_val_acc > best_val:
            best_val = best_val_acc
            best_fold_artifacts = fold_summaries[-1]

    # Save CV summary
    with open(os.path.join(OUT_DIR, "cv_summary.json"), "w") as f:
        json.dump({"folds": fold_summaries}, f, indent=2)

    print("\nBest fold by VAL accuracy:", best_fold_artifacts)
    return fold_summaries, best_fold_artifacts

# -----------------------
# 10) Main
# -----------------------
if __name__ == "__main__":
    fold_summaries, best = run_cv()
    # Run explainability for the best fold on 5 imgs/class from its TEST set
    run_explainability_v2(best_fold_dir=best["dir"], best_cnn_path=best["cnn_ckpt"],
                          test_indices=np.array(best["test_indices"]))

    print("\nAll done. Check:", OUT_DIR)


In [ ]:
import os
import glob
import math
import itertools
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.preprocessing import image as kimage
from tensorflow.keras.utils import to_categorical

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# -----------------------
# 0) Paths & params
# -----------------------
data_dir = r"D:/Customised_CNN/dataset/Brain_Tumor/four_class"  # your dataset
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

# GCN params
K = 10
LR_GCN = 1e-2
WD = 5e-4
EPOCHS_GCN = 200
HIDDEN = 16
DROPOUT = 0.5

# CNN params (your architecture)
LR_CNN = 1e-3
EPOCHS_CNN = 20
VAL_SPLIT = 0.2
TEST_SIZE = 0.2

# -----------------------
# 1) Index files & labels
# -----------------------
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c:i for i,c in enumerate(class_names)}

paths = []
labels_int = []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int)
num_classes = len(class_names)
print(f"Found {len(paths)} images across {num_classes} classes: {class_names}")

# Split indices (stratified)
all_idx = np.arange(len(paths))
idx_tr, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_tr, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_tr])

mask_tr = np.zeros(len(paths), dtype=bool); mask_tr[idx_tr] = True
mask_va = np.zeros(len(paths), dtype=bool); mask_va[idx_va] = True
mask_te = np.zeros(len(paths), dtype=bool); mask_te[idx_te] = True

# -----------------------
# 2) TF datasets to train your CNN
# -----------------------
def load_img(path, label):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    return arr, label

# -----------------------
# REPLACE your make_dataset() + load_img() with this
# -----------------------
IMG_SIZE = (150, 150)  # already set above

def load_and_preprocess(path, label):
    # Read file
    img = tf.io.read_file(path)
    # Decode (handles jpg/png/bmp; expand_animations=False to avoid GIF frames)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    # Convert to float32 [0,1]
    img = tf.image.convert_image_dtype(img, tf.float32)
    # Resize to target
    img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    # Set static shape so Keras knows the ranks/sizes
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    # Ensure label dtype is int32 for sparse CE
    label = tf.cast(label, tf.int32)
    return img, label

def make_dataset(indexes, shuffle=True, repeat=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((paths[indexes], labels_int[indexes]))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indexes), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    if repeat:
        ds = ds.repeat()
    return ds

# Rebuild datasets
train_ds = make_dataset(idx_tr, shuffle=True,  batch_size=BATCH_SIZE)
val_ds   = make_dataset(idx_va, shuffle=False, batch_size=BATCH_SIZE)
test_ds  = make_dataset(idx_te, shuffle=False, batch_size=BATCH_SIZE)

# For feature extraction later we’ll also need ALL images in fixed order:
all_ds   = make_dataset(all_idx, shuffle=False, batch_size=BATCH_SIZE)

# -----------------------
# 3) Your CNN (as requested)
#    We'll NAME the penultimate Dense(512) layer "feat" so we can extract embeddings.
# -----------------------
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (4, 4), activation="relu", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(64, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.MaxPooling2D(pool_size=(3, 3)),
        layers.Conv2D(128, (4, 4), activation="relu"),
        layers.Flatten(),
        layers.Dense(512, activation="relu", name="feat"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

cnn = create_model()
cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

print(cnn.summary())
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN, verbose=1)

# -----------------------
# 4) Build embeddings for ALL images using the trained CNN up to "feat"
# -----------------------
feat_extractor = Model(inputs=cnn.input, outputs=cnn.get_layer("feat").output)
def extract_features(ds):
    feats = []
    ys = []
    for xb, yb in ds:
        f = feat_extractor(xb, training=False).numpy()
        feats.append(f); ys.append(yb.numpy())
    return np.vstack(feats), np.concatenate(ys)

# Create a dataset over ALL images in original order to preserve index alignment
all_ds = make_dataset(all_idx, shuffle=False)
X_features, y_ordered = extract_features(all_ds)  # y_ordered == labels_int (sanity check)
assert np.all(y_ordered == labels_int), "Label order mismatch!"

X = X_features.astype(np.float32)                         # (N, F)
y = to_categorical(labels_int, num_classes=num_classes)   # (N, C)
N, F = X.shape

# -----------------------
# 5) kNN graph over features
# -----------------------
nbrs = NearestNeighbors(n_neighbors=K+1, metric="euclidean").fit(X)
_, knn_idx = nbrs.kneighbors(X, return_distance=True)

rows, cols, data = [], [], []
for i in range(N):
    for j in knn_idx[i]:
        if i == j:  # skip self; will add self loops later
            continue
        rows.append(i); cols.append(j); data.append(1.0)

A = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
A = A.maximum(A.T)          # symmetrize
A.setdiag(1.0)              # self loops
A = A.tocsr()
A_norm = gcn_filter(A)      # ĨD^{-1/2} Ĩ D^{-1/2}

# -----------------------
# [STEP 5A] Paper visuals
# -----------------------
# t-SNE (2D) — may take time; sample if very large
from sklearn.manifold import TSNE
print("[VIS] Running t-SNE (2D) on features...")
if N > 4000:
    # sample for speed
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(N, size=4000, replace=False)
    X_ts = X_std[sample_idx]
    y_ts = labels_int[sample_idx]
else:
    X_ts, y_ts = X_std, labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = (y_ts == i)
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=6, alpha=0.7, label=cname)
plt.title("t-SNE of Feature Embeddings")
plt.xlabel("t-SNE-1"); plt.ylabel("t-SNE-2")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.savefig("fig_tsne_features.png", dpi=300)
plt.show()

# Degree histogram
plt.figure(figsize=(7, 4))
plt.hist(degrees, bins=range(int(degrees.min()), int(degrees.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node degree (excluding self-loop)")
plt.ylabel("Count")
plt.title("Degree Distribution of Mutual-kNN Graph")
plt.tight_layout()
plt.savefig("fig_degree_hist.png", dpi=300)
plt.show()

# Adjacency snapshot (top-left block)
subN = min(150, N)
A_small = A_mut[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest")
plt.title(f"Adjacency (top-left {subN}×{subN})")
plt.xlabel("Node index"); plt.ylabel("Node index")
plt.tight_layout()
plt.savefig("fig_adjacency_snapshot.png", dpi=300)
plt.show()

# -----------------------
# [STEP 5B] OPTIONAL — one per-class image→pixel-graph (32×32)
# -----------------------
try:
    import cv2, networkx as nx
    from spektral.data import Graph as SpGraph
    px_image_size = (32, 32)

    def px_load_images(data_dir, image_size=(32, 32), classes=None):
        if classes is None:
            classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        imgs, lbls = [], []
        for cname in classes:
            cdir = os.path.join(data_dir, cname)
            if not os.path.isdir(cdir): 
                continue
            for f in os.listdir(cdir):
                p = os.path.join(cdir, f)
                img = cv2.imread(p, cv2.IMREAD_COLOR)
                if img is None:
                    continue
                if img.shape[-1] == 4:
                    img = img[:, :, :3]
                img = cv2.resize(img, image_size, interpolation=cv2.INTER_AREA)
                imgs.append(img); lbls.append(cname)
        name_to_id = {n: i for i, n in enumerate(classes)}
        lbls_int = np.array([name_to_id[s] for s in lbls], dtype=np.int32)
        return np.array(imgs), lbls_int, classes

    def image_to_graph(img):
        h, w, c = img.shape
        n = h * w
        x = (img.reshape(n, c).astype(np.float32)) / 255.0
        a = np.zeros((n, n), dtype=np.float32)
        for i in range(h):
            for j in range(w):
                idx = i * w + j
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < h and 0 <= nj < w:
                            a[idx, ni * w + nj] = 1.0
        return SpGraph(x=x, a=a)

    def visualize_image_and_graph(img_bgr, graph, title, savepath=None):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        G = nx.from_numpy_array(graph.a)
        pos = nx.spring_layout(G, seed=42)
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
        axes[0].imshow(img_rgb); axes[0].set_title(f"{title} — 32×32 image"); axes[0].axis("off")
        nx.draw(G, pos, node_size=12, node_color=graph.x[:, 0], cmap="viridis", with_labels=False, ax=axes[1])
        axes[1].set_title("8-neighbor pixel graph"); axes[1].axis("off")
        plt.tight_layout()
        if savepath: plt.savefig(savepath, dpi=300)
        plt.show()

    px_imgs, px_labels, px_classes = px_load_images(data_dir, image_size=px_image_size, classes=class_names)
    selected_idxs = []
    for cls_id, cname in enumerate(px_classes):
        idxs = np.where(px_labels == cls_id)[0]
        if idxs.size == 0:
            print(f"[WARN] No samples for '{cname}'"); 
            continue
        selected_idxs.append(int(idxs[0]))

    for i in selected_idxs:
        g = image_to_graph(px_imgs[i])
        cname = px_classes[px_labels[i]]
        visualize_image_and_graph(px_imgs[i], g, title=f"{cname}", savepath=f"fig_pixelgraph_oneperclass_{cname}.png")

except Exception as e:
    print(f"[STEP 5B] Skipped pixel-graph viz: {e}")

# -----------------------
# 6) GCN model (2 layers)
# -----------------------
X_in = Input(shape=(F,), name="X_in")
A_in = Input((N,), sparse=True, name="A_in")

h = GCNConv(HIDDEN, activation="relu",
            kernel_regularizer=regularizers.l2(WD))([X_in, A_in])
h = layers.Dropout(DROPOUT)(h)
out = GCNConv(num_classes, activation="softmax")([h, A_in])

gcn = Model(inputs=[X_in, A_in], outputs=out)
gcn.compile(optimizer=tf.keras.optimizers.Adam(LR_GCN),
            loss="categorical_crossentropy",
            weighted_metrics=["accuracy"])

hist_gcn = gcn.fit(
    x=[X, A_norm],
    y=y,
    sample_weight=mask_tr.astype(np.float32),
    batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1,
    validation_data=([X, A_norm], y, mask_va.astype(np.float32))
)
# -----------------------
# 7) Evaluate on VALIDATION (GCN)
# -----------------------
val_loss, val_acc = gcn.evaluate(
    x=[X, A_norm], y=y,
    sample_weight=mask_va.astype(np.float32),
    batch_size=N, verbose=0
)
print(f"[GCN] Validation accuracy: {val_acc:.4f}  |  Validation loss: {val_loss:.4f}")

# -----------------------
# 8) Reports: classification report, confusion matrix, ROC (GCN) on VALIDATION
# -----------------------
y_prob = gcn.predict([X, A_norm], batch_size=N, verbose=0)   # (N, C)
y_pred = np.argmax(y_prob, axis=1)
y_true = labels_int

print("\n[GCN] Classification Report (VALIDATION):")
print(classification_report(y_true[mask_va], y_pred[mask_va], target_names=class_names))

cm = confusion_matrix(y_true[mask_va], y_pred[mask_va])
print("[GCN] Confusion Matrix (VALIDATION):\n", cm)

# Confusion matrix plot (VALIDATION)
plt.figure(figsize=(6, 6))
plt.imshow(cm, cmap="Blues")
plt.title("GCN Confusion Matrix (Validation)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.show()

# ROC curves (one-vs-rest) on VALIDATION
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_val_bin  = label_binarize(y_true[mask_va], classes=np.arange(num_classes))
y_pred_bin = y_prob[mask_va]

fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_val_bin[:, i], y_pred_bin[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average
fpr["micro"], tpr["micro"], _ = roc_curve(y_val_bin.ravel(), y_pred_bin.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8, 6))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("GCN ROC Curves (Validation)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

# -----------------------
# 9) Training curves (Loss & Accuracy) — already reflect validation during training
# -----------------------
def plot_curves(history, title_prefix=""):
    acc_key = "accuracy" if "accuracy" in history.history else "acc"
    val_acc_key = "val_accuracy" if "val_accuracy" in history.history else "val_acc"
    plt.figure(figsize=(12,5))
    # Loss
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="Train Loss")
    if "val_loss" in history.history:
        plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{title_prefix} Loss"); plt.legend()
    # Accuracy
    plt.subplot(1,2,2)
    if acc_key in history.history:
        plt.plot(history.history[acc_key], label="Train Acc")
    if val_acc_key in history.history:
        plt.plot(history.history[val_acc_key], label="Val Acc")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"{title_prefix} Accuracy"); plt.legend()
    plt.tight_layout(); plt.show()

plot_curves(hist_gcn, title_prefix="GCN")
# If you want the CNN curves too:
# plot_curves(hist_cnn, title_prefix="CNN")

# -----------------------
# 7) Evaluate on TEST (GCN)
# -----------------------
test_loss, test_acc = gcn.evaluate(
    x=[X, A_norm], y=y,
    sample_weight=mask_te.astype(np.float32),
    batch_size=N, verbose=0
)
print(f"[GCN] Test accuracy: {test_acc:.4f}")

# -----------------------
# 8) Reports: classification report, confusion matrix, ROC (GCN)
# -----------------------
y_prob = gcn.predict([X, A_norm], batch_size=N, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = labels_int

print("\n[GCN] Classification Report (TEST):")
print(classification_report(y_true[mask_te], y_pred[mask_te], target_names=class_names))

cm = confusion_matrix(y_true[mask_te], y_pred[mask_te])
print("[GCN] Confusion Matrix:\n", cm)

# Confusion matrix plot
plt.figure(figsize=(6, 6))
plt.imshow(cm, cmap="Blues")
plt.title("GCN Confusion Matrix (Test)")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
plt.tight_layout(); plt.show()

# ROC curves (one-vs-rest)
y_test_bin = label_binarize(y_true[mask_te], classes=np.arange(num_classes))
y_pred_bin = y_prob[mask_te]

fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_bin[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# micro-average
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_pred_bin.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8, 6))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("GCN ROC Curves (Test)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

# -----------------------
# 9) Training curves (Loss & Accuracy) — GCN (and CNN optional)
# -----------------------
def plot_curves(history, title_prefix=""):
    # Handle TF key variants
    acc_key = "accuracy" if "accuracy" in history.history else "acc"
    val_acc_key = "val_accuracy" if "val_accuracy" in history.history else "val_acc"
    plt.figure(figsize=(12,5))
    # Loss
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="Train Loss")
    if "val_loss" in history.history:
        plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{title_prefix} Loss"); plt.legend()
    # Accuracy
    plt.subplot(1,2,2)
    if acc_key in history.history:
        plt.plot(history.history[acc_key], label="Train Acc")
    if val_acc_key in history.history:
        plt.plot(history.history[val_acc_key], label="Val Acc")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"{title_prefix} Accuracy"); plt.legend()
    plt.tight_layout(); plt.show()

plot_curves(hist_gcn, title_prefix="GCN")
# If you want to see the CNN curves, uncomment:
# plot_curves(hist_cnn, title_prefix="CNN")


In [ ]:
import os, cv2, shap
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries
from lime.lime_image import LimeImageExplainer

os.makedirs("explain_out/lime", exist_ok=True)
os.makedirs("explain_out/shap", exist_ok=True)
os.makedirs("explain_out/gradcam/correct", exist_ok=True)
os.makedirs("explain_out/gradcam/misclassified", exist_ok=True)

def materialize_dataset(ds):
    xs, ys = [], []
    # defensive: ds may already be batched; unbatch then re-batch to avoid huge memory spikes
    for xb, yb in ds.unbatch().batch(256):
        xs.append(xb.numpy())
        ys.append(yb.numpy())
    X = np.concatenate(xs, axis=0)   # float32 in [0,1] with shape (N,150,150,3)
    y = np.concatenate(ys, axis=0)   # int32 labels
    return X, y

# pull arrays for XAI
X_train_np, y_train_np = materialize_dataset(train_ds)
X_val_np,   y_val_np   = materialize_dataset(val_ds)
X_test_np,  y_test_np  = materialize_dataset(test_ds)

# convenience versions for LIME (expects uint8)
X_test_uint8 = (np.clip(X_test_np * 255.0, 0, 255)).astype(np.uint8)

# predictions from your trained CNN
y_prob_test = cnn.predict(X_test_np, verbose=0)
y_pred_test = np.argmax(y_prob_test, axis=1)

# class name map
id2name = {i: n for i, n in enumerate(class_names)}


In [ ]:
def generate_lime_explanations_like_image(model, X_samples_uint8, y_samples_int, id2name, 
                                          num_samples=1000, num_features=10):
    explainer = LimeImageExplainer()

    def predict_fn(images):
        # images may be uint8 [0..255] or float; normalize to [0,1], resize to IMG_SIZE
        x = tf.convert_to_tensor(images, dtype=tf.float32)
        if tf.reduce_max(x) > 1.0:
            x = x / 255.0
        x = tf.image.resize(x, IMG_SIZE)  # safety; your data is already 150x150
        return model(x, training=False).numpy()

    fig, axs = plt.subplots(len(X_samples_uint8), 3, figsize=(15, 5 * len(X_samples_uint8)))
    if len(X_samples_uint8) == 1:
        axs = np.array([axs])  # normalize indexing

    for i in range(len(X_samples_uint8)):
        explanation = explainer.explain_instance(
            image=X_samples_uint8[i].astype("double"),
            classifier_fn=predict_fn,
            top_labels=1,
            hide_color=0,
            num_samples=num_samples
        )
        # predicted label for this sample
        pred_label = int(np.argmax(predict_fn([X_samples_uint8[i]])[0]))

        temp, mask = explanation.get_image_and_mask(
            label=pred_label,
            positive_only=False,
            num_features=num_features,
            hide_rest=False
        )

        black = np.zeros_like(temp)
        colored = np.zeros_like(temp)
        colored[mask == 1]  = [0, 255, 0]   # positive → green
        colored[mask == -1] = [255, 0, 0]   # negative → red

        axs[i, 0].imshow(X_samples_uint8[i]); axs[i, 0].set_title(f'Input: {id2name[int(y_samples_int[i])]}')
        axs[i, 0].axis('off')

        axs[i, 1].imshow(black); axs[i, 1].imshow(colored, alpha=0.6)
        axs[i, 1].set_title('Features (Green=+ , Red=−)'); axs[i, 1].axis('off')

        axs[i, 2].imshow(X_samples_uint8[i])
        axs[i, 2].imshow(mark_boundaries(temp / 255.0, mask), alpha=0.6)
        axs[i, 2].set_title(f'Output: {id2name[pred_label]}'); axs[i, 2].axis('off')

    plt.tight_layout(); plt.suptitle("LIME Explanations", fontsize=16, y=1.02)
    plt.savefig("explain_out/lime/lime_grid.png", dpi=180, bbox_inches='tight')
    plt.show()

# pick up to 5 per-class from test for LIME
sel_idx = []
per_class = {c: 0 for c in range(len(class_names))}
for i, y in enumerate(y_test_np):
    if per_class[int(y)] < 5:
        per_class[int(y)] += 1
        sel_idx.append(i)
    if all(v >= 5 for v in per_class.values()):
        break

generate_lime_explanations_like_image(
    model=cnn,
    X_samples_uint8=X_test_uint8[sel_idx],
    y_samples_int=y_test_np[sel_idx],
    id2name=id2name,
    num_samples=1000,
    num_features=10
)


In [ ]:
import os, cv2, numpy as np
os.makedirs("explain_out/shap/per_image", exist_ok=True)

def add_title_bar(img_bgr: np.ndarray, text: str,
                  font=cv2.FONT_HERSHEY_SIMPLEX, font_scale=0.55, thickness=1) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    bar_h = 30
    canvas = np.full((bar_h + h, w, 3), 255, dtype=np.uint8)
    canvas[bar_h:] = img_bgr
    # Trim very long names
    max_chars = max(12, int(w / 7))
    if len(text) > max_chars:
        text = text[:max_chars - 3] + "..."
    # color code CORRECT / WRONG at the end if present
    color = (0, 0, 0)
    if text.endswith("[CORRECT]"):
        color = (0, 128, 0)
    elif text.endswith("[WRONG]"):
        color = (0, 0, 180)
    cv2.putText(canvas, text, (8, int(bar_h * 0.75)), font, font_scale, color, thickness, cv2.LINE_AA)
    return canvas


In [ ]:
# filenames aligned with test order (since test_ds was built with shuffle=False)
test_files = [os.path.basename(p) for p in paths[idx_te]]

def save_shap_overlays_with_titles(explainer,             # your 'ex'
                                   model_cnn,             # your CNN (for fallback pred)
                                   X_test_uint8,          # uint8 [0..255]
                                   X_test_float01,        # float [0,1]
                                   y_true_idx,            # int labels
                                   file_names,            # test_files from above
                                   id2name,               # {i: class_name}
                                   pred_final_idx=None,   # optional: ensemble predictions (int array)
                                   out_dir="explain_out/shap/per_image"):
    os.makedirs(out_dir, exist_ok=True)

    for i in range(len(X_test_uint8)):
        # choose which label to explain: final pred (ensemble) if available, else CNN pred
        if pred_final_idx is not None:
            pred_idx = int(pred_final_idx[i])
        else:
            # CNN fallback
            p = model_cnn.predict(X_test_float01[i:i+1], verbose=0)
            pred_idx = int(np.argmax(p, axis=1)[0])

        # run SHAP for THIS image & THIS output only (keeps plots meaningful)
        exp_i = explainer(X_test_uint8[i:i+1], outputs=[pred_idx], max_evals=500, batch_size=32)

        # turn SHAP values into a heatmap (mean |attribution| across RGB)
        # shapes are (1, H, W, 3)
        vals = exp_i.values[0]                    # (H, W, 3)
        heat = np.mean(np.abs(vals), axis=-1)     # (H, W)
        heat = heat / (heat.max() + 1e-8)

        # colorize and overlay on the original image
        base_rgb = (X_test_float01[i] * 255.0).astype(np.uint8)     # RGB
        heat_u8  = (heat * 255).astype(np.uint8)
        heatmap  = cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET)     # BGR
        overlay  = cv2.addWeighted(base_rgb[:, :, ::-1], 0.55, heatmap, 0.45, 0)  # to BGR then blend

        # build title
        true_name = id2name[int(y_true_idx[i])]
        pred_name = id2name[int(pred_idx)]
        ok = "[CORRECT]" if int(y_true_idx[i]) == pred_idx else "[WRONG]"
        title = f"{file_names[i]} | True: {true_name} | Pred: {pred_name} {ok}"

        # add title bar & save (BGR)
        out_img = add_title_bar(overlay, title)
        out_path = os.path.join(out_dir, f"{i:04d}_{ok[1:-1]}_{os.path.splitext(file_names[i])[0]}.png")
        cv2.imwrite(out_path, out_img)


In [ ]:
# ===== 1) filenames aligned to test order =====
import os, cv2, numpy as np

test_files = [os.path.basename(p) for p in paths[idx_te]]
id2name = {i: n for i, n in enumerate(class_names)}

# ===== 2) helper: add a title bar to a BGR image =====
def add_title_bar(img_bgr: np.ndarray, text: str,
                  font=cv2.FONT_HERSHEY_SIMPLEX, font_scale=0.55, thickness=1) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    bar_h = 30
    canvas = np.full((bar_h + h, w, 3), 255, dtype=np.uint8)
    canvas[bar_h:] = img_bgr
    max_chars = max(12, int(w / 7))
    if len(text) > max_chars:
        text = text[:max_chars - 3] + "..."
    color = (0, 0, 0)
    if text.endswith("[CORRECT]"):
        color = (0, 128, 0)
    elif text.endswith("[WRONG]"):
        color = (0, 0, 180)
    cv2.putText(canvas, text, (8, int(bar_h * 0.75)), font, font_scale, color, thickness, cv2.LINE_AA)
    return canvas

# ===== 3) pick 5 per class (or all if fewer available) =====
def pick_k_per_class(y_int: np.ndarray, k: int, num_classes: int, seed: int = 123):
    sel = []
    rng = np.random.default_rng(seed)
    for c in range(num_classes):
        idxs = np.where(y_int == c)[0]
        if idxs.size == 0:
            continue
        if idxs.size > k:
            chosen = rng.choice(idxs, size=k, replace=False)
        else:
            chosen = idxs
        sel.extend(sorted(chosen))
    return sorted(sel)

sel_idx = pick_k_per_class(y_test_np, k=5, num_classes=len(class_names), seed=SEED)

# ===== 4) SHAP overlays with filename + truth + final prediction =====
import os
os.makedirs("explain_out/shap/per_image_5perclass", exist_ok=True)

def save_shap_overlays_with_titles_subset(explainer,             # your 'ex'
                                          model_cnn,             # CNN (fallback for preds)
                                          X_test_uint8,          # uint8 [0..255]
                                          X_test_float01,        # float [0,1]
                                          y_true_idx,            # ints
                                          file_names,            # aligned filenames
                                          id2name,               # {i: name}
                                          indices,               # subset indices (5 per class)
                                          pred_final_idx=None,   # optional ensemble preds (ints, full test set)
                                          out_dir="explain_out/shap/per_image_5perclass"):
    for i in indices:
        # FINAL prediction: ensemble if provided, else CNN
        if pred_final_idx is not None:
            pred_idx = int(pred_final_idx[i])
        else:
            p = model_cnn.predict(X_test_float01[i:i+1], verbose=0)
            pred_idx = int(np.argmax(p, axis=1)[0])

        # explain THIS image for THIS predicted class
        exp_i = explainer(X_test_uint8[i:i+1], outputs=[pred_idx], max_evals=500, batch_size=32)

        # build a heatmap from SHAP values
        vals = exp_i.values[0]                  # (H,W,3)
        heat = np.mean(np.abs(vals), axis=-1)   # (H,W)
        heat = heat / (heat.max() + 1e-8)
        heat_u8 = (heat * 255).astype(np.uint8)
        heatmap = cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET)  # BGR

        base_rgb = (X_test_float01[i] * 255.0).astype(np.uint8)
        overlay  = cv2.addWeighted(base_rgb[:, :, ::-1], 0.55, heatmap, 0.45, 0)  # to BGR then blend

        # title
        true_name = id2name[int(y_true_idx[i])]
        pred_name = id2name[pred_idx]
        ok = "[CORRECT]" if int(y_true_idx[i]) == pred_idx else "[WRONG]"
        title = f"{file_names[i]} | True: {true_name} | Pred: {pred_name} {ok}"

        # save
        out_img = add_title_bar(overlay, title)
        out_path = os.path.join(out_dir, f"{i:04d}_{ok[1:-1]}_{os.path.splitext(file_names[i])[0]}.png")
        cv2.imwrite(out_path, out_img)

# ===== 5) call it =====
# If you have ensemble preds on TEST (recommended):
# y_ens_test = best_alpha * y_prob_cnn_all[mask_te] + (1 - best_alpha) * y_prob_gcn_all[mask_te]
# pred_final = np.argmax(y_ens_test, axis=1)

# With ensemble:
# save_shap_overlays_with_titles_subset(ex, cnn, X_test_uint8, X_test_np, y_test_np,
#                                       test_files, id2name, sel_idx, pred_final_idx=pred_final)

# Or CNN-only:
save_shap_overlays_with_titles_subset(ex, cnn, X_test_uint8, X_test_np, y_test_np,
                                      test_files, id2name, sel_idx, pred_final_idx=None)


In [ ]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt

# Filenames aligned with TEST order (since test_ds was created with shuffle=False)
test_files = [os.path.basename(p) for p in paths[idx_te]]
id2name = {i: n for i, n in enumerate(class_names)}

# If you don't already have uint8 copies for SHAP masker, make them:
X_test_uint8 = (np.clip(X_test_np * 255.0, 0, 255)).astype(np.uint8)  # RGB

def pick_k_per_class(y_int: np.ndarray, k: int, num_classes: int, seed: int = 123):
    sel = []
    rng = np.random.default_rng(seed)
    for c in range(num_classes):
        idxs = np.where(y_int == c)[0]
        if idxs.size == 0:
            continue
        choose = idxs if idxs.size <= k else rng.choice(idxs, size=k, replace=False)
        sel.extend(sorted(choose))
    return sorted(sel)

def display_shap_5_per_class(explainer, model_cnn, X_uint8, X_float01, y_true_int,
                             file_names, class_names, pred_final_idx=None, k=5, seed=123):
    num_classes = len(class_names)
    # pick up to k indices for each class (based on TRUE labels)
    per_class_idx = {c: [] for c in range(num_classes)}
    for c in range(num_classes):
        idxs = np.where(y_true_int == c)[0]
        if idxs.size == 0: 
            continue
        rng = np.random.default_rng(seed)
        if idxs.size > k:
            idxs = np.sort(rng.choice(idxs, size=k, replace=False))
        per_class_idx[c] = idxs.tolist()

    for c in range(num_classes):
        idxs = per_class_idx.get(c, [])
        if not idxs:
            continue

        cols = len(idxs)
        fig, axes = plt.subplots(1, cols, figsize=(4.2 * cols, 4.4))
        if cols == 1:
            axes = [axes]

        for ax, i in zip(axes, idxs):
            # final prediction: ensemble if available, else CNN
            if pred_final_idx is not None:
                pred_idx = int(pred_final_idx[i])
            else:
                prob = model_cnn.predict(X_float01[i:i+1], verbose=0)
                pred_idx = int(np.argmax(prob, axis=1)[0])

            # SHAP for this image *and* this predicted class (keeps plot focused)
            exp_i = explainer(X_uint8[i:i+1], outputs=[pred_idx], max_evals=500, batch_size=32)

            # convert SHAP values to heatmap and overlay
            vals = exp_i.values[0]                  # (H, W, 3)
            heat = np.mean(np.abs(vals), axis=-1)   # (H, W)
            heat = heat / (heat.max() + 1e-8)
            heat_u8 = (heat * 255).astype(np.uint8)
            heatmap = cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET)        # BGR
            base_bgr = (X_float01[i] * 255.0).astype(np.uint8)[:, :, ::-1]# to BGR
            overlay  = cv2.addWeighted(base_bgr, 0.55, heatmap, 0.45, 0)
            overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

            ok = "CORRECT" if int(y_true_int[i]) == pred_idx else "WRONG"
            title = f"{file_names[i]}\nTrue: {class_names[y_true_int[i]]} | Pred: {class_names[pred_idx]} [{ok}]"

            ax.imshow(overlay_rgb)
            ax.set_title(title, fontsize=9)
            ax.axis('off')

        plt.suptitle(f"SHAP overlays — class {class_names[c]} (up to {k})", y=1.02, fontsize=12)
        plt.tight_layout()
        plt.show()


In [ ]:
# y_prob_cnn_all = cnn.predict(all_ds, verbose=0)
# y_prob_gcn_all = gcn.predict([X, A_norm], batch_size=N, verbose=0)
# best_alpha chosen on validation as you already do
y_ens_test = best_alpha * y_prob_cnn_all[mask_te] + (1.0 - best_alpha) * y_prob_gcn_all[mask_te]
pred_final = np.argmax(y_ens_test, axis=1)

display_shap_5_per_class(ex, cnn, X_test_uint8, X_test_np, y_test_np,
                         test_files, class_names, pred_final_idx=pred_final, k=5, seed=SEED)


In [ ]:
display_shap_5_per_class(ex, cnn, X_test_uint8, X_test_np, y_test_np,
                         test_files, class_names, pred_final_idx=None, k=5, seed=SEED)


In [ ]:
save_shap_overlays_with_titles(
    explainer=ex,
    model_cnn=cnn,
    X_test_uint8=X_test_uint8,
    X_test_float01=X_test_np,
    y_true_idx=y_test_np,
    file_names=test_files,
    id2name=id2name,
    pred_final_idx=None,                   # <- use CNN prediction
    out_dir="explain_out/shap/per_image"
)


In [ ]:
# Build ensemble predictions on TEST if you haven’t already
# y_prob_cnn_all = cnn.predict(all_ds, verbose=0)
# y_prob_gcn_all = gcn.predict([X, A_norm], batch_size=N, verbose=0)
# (choose best_alpha on validation as you did)
y_ens_test = best_alpha * y_prob_cnn_all[mask_te] + (1.0 - best_alpha) * y_prob_gcn_all[mask_te]
pred_final = np.argmax(y_ens_test, axis=1)

id2name = {i: n for i, n in enumerate(class_names)}
save_shap_overlays_with_titles(
    explainer=ex,
    model_cnn=cnn,
    X_test_uint8=X_test_uint8,
    X_test_float01=X_test_np,
    y_true_idx=y_test_np,
    file_names=test_files,
    id2name=id2name,
    pred_final_idx=pred_final,             # <- ensemble final prediction
    out_dir="explain_out/shap/per_image"
)


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import shap, tensorflow as tf

# filenames aligned to TEST order (since test_ds was built with shuffle=False)
test_files = [os.path.basename(p) for p in paths[idx_te]]
id2name = {i: n for i, n in enumerate(class_names)}

# ensure uint8 images for the Image masker; float [0,1] for model preds
X_test_uint8 = (np.clip(X_test_np * 255.0, 0, 255)).astype(np.uint8)

def pick_k_per_class(y_int, k=5, num_classes=None, seed=123):
    if num_classes is None:
        num_classes = int(np.max(y_int)) + 1
    rng = np.random.default_rng(seed)
    out = []
    for c in range(num_classes):
        idxs = np.where(y_int == c)[0]
        if idxs.size == 0: 
            continue
        if idxs.size > k:
            idxs = np.sort(rng.choice(idxs, size=k, replace=False))
        out.extend(idxs.tolist())
    return out

def display_shap_5_per_class_color(explainer,           # your explainer built with output_names=class_names
                                   model_cnn,           # CNN for fallback preds
                                   X_uint8, X_float01,  # uint8 for SHAP; float[0,1] for model
                                   y_true_int,          # int labels
                                   file_names,          # filenames aligned to test order
                                   class_names,         # list of names
                                   pred_final_idx=None, # optional: ensemble predictions (int array for test)
                                   k=5, seed=123):
    C = len(class_names)
    for c in range(C):
        idxs_all = np.where(y_true_int == c)[0]
        if idxs_all.size == 0:
            continue
        # pick up to k
        rng = np.random.default_rng(seed)
        idxs = np.sort(rng.choice(idxs_all, size=min(k, idxs_all.size), replace=False))

        # explain these samples for all outputs so columns are per class (SHAP red/blue)
        exp = explainer(X_uint8[idxs], outputs=range(C), max_evals=500, batch_size=32)

        # SHAP's native plot (red↔blue diverging colormap)
        plt.figure(figsize=(12, 4 * len(idxs)))
        shap.image_plot(exp, show=False)   # <- keeps SHAP's colors and column headers

        # annotate the first column (original image) with filename + truth + final prediction
        ncols = C + 1                      # image column + one per class
        axes = plt.gcf().axes
        for r, i in enumerate(idxs):
            if pred_final_idx is not None:
                pred_idx = int(pred_final_idx[i])
            else:
                prob = model_cnn.predict(X_float01[i:i+1], verbose=0)
                pred_idx = int(np.argmax(prob, axis=1)[0])

            ok = "CORRECT" if pred_idx == int(y_true_int[i]) else "WRONG"
            title = f"{file_names[i]}\nTrue: {class_names[int(y_true_int[i])]} | " \
                    f"Pred: {class_names[pred_idx]} [{ok}]"

            # first column axis for this row
            ax0 = axes[r * ncols]
            ax0.set_title(title, fontsize=9)

        plt.tight_layout()
        plt.show()

# ---- call it ----
# If you have ensemble predictions on TEST:
# y_ens_test = best_alpha * y_prob_cnn_all[mask_te] + (1.0 - best_alpha) * y_prob_gcn_all[mask_te]
# pred_final  = np.argmax(y_ens_test, axis=1)

# With ensemble:
# display_shap_5_per_class_color(ex, cnn, X_test_uint8, X_test_np, y_test_np,
#                                test_files, class_names, pred_final_idx=pred_final, k=5, seed=SEED)

# Or CNN-only:
display_shap_5_per_class_color(ex, cnn, X_test_uint8, X_test_np, y_test_np,
                               test_files, class_names, pred_final_idx=None, k=5, seed=SEED)


In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model

# --- 1) Build a "logits model" from your trained CNN (removes softmax) ---
# Assumes your CNN has a penultimate layer named "feat" and final Dense softmax
last_dense = cnn.layers[-1]                     # Dense(..., activation='softmax')
W, b = last_dense.get_weights()                 # copy softmax layer weights
feat_model = Model(inputs=cnn.input, outputs=cnn.get_layer("feat").output)

# linear logits layer with identical weights
logits_layer = layers.Dense(W.shape[1], activation=None, name="logits")
logits = logits_layer(feat_model.output)
logits_model = Model(inputs=cnn.input, outputs=logits)
logits_layer.set_weights([W, b])                # set copied weights

# --- 2) SHAP explainer with an image blur masker baseline ---
# SHAP's Image masker expects uint8-like scale. We'll feed uint8 and normalize inside f().
masker = shap.maskers.Image("blur(16,16)", (IMG_SIZE[0], IMG_SIZE[1], 3))

def f(images_uint8):
    # images may be uint8 [0..255] or float; normalize to [0,1] & resize
    x = tf.convert_to_tensor(images_uint8, dtype=tf.float32)
    if tf.reduce_max(x) > 1.0:
        x = x / 255.0
    x = tf.image.resize(x, IMG_SIZE)
    # use logits for attribution, but return probabilities for nicer class names if desired
    logits = logits_model(x, training=False)
    probs  = tf.nn.softmax(logits)              # not used by SHAP math, just for output labels
    return probs.numpy()

# --- 3) Choose a small explanation set + run SHAP ---
# Reuse the same selection you used for LIME, or pick 8 from test:
sel = np.arange(min(8, len(X_test_uint8)))     # X_test_uint8 from earlier: uint8 [0..255]
ex = shap.Explainer(f, masker, output_names=class_names)

vals = ex(X_test_uint8[sel], max_evals=500, batch_size=32)

print("max |SHAP|:", np.max(np.abs(vals.values)))  # sanity check: should be >> 1e-7 now

# --- 4) Plot & save ---
plt.figure(figsize=(10, 6))
shap.image_plot(vals, show=False)               # new API
plt.tight_layout()
plt.savefig("explain_out/shap/shap_summary_logit_masker.png", dpi=180)
plt.show()


In [ ]:
def get_last_conv_layer_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found in model.")

LAST_CONV = get_last_conv_layer_name(cnn)

def make_gradcam_heatmap(img_float01, model, last_conv_layer_name, pred_index=None):
    img_b = np.expand_dims(img_float01, axis=0)
    grad_model = tf.keras.models.Model([model.inputs],
                                       [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_b, training=False)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        loss = preds[:, pred_index]
    grads = tape.gradient(loss, conv_out)[0]
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    cam = tf.reduce_sum(weights * conv_out[0], axis=-1)
    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], IMG_SIZE)
    return cam.numpy().squeeze()

def generate_gradcam_for_images(model, X_float01, y_true_int, y_pred_int, id2name, last_conv, out_dir, num_each=5):
    # pick some correct & misclassified
    correct_idx = np.where(y_true_int == y_pred_int)[0][:num_each]
    mis_idx     = np.where(y_true_int != y_pred_int)[0][:num_each]

    for tag, idxs in [("correct", correct_idx), ("misclassified", mis_idx)]:
        for i, idx in enumerate(idxs):
            img = X_float01[idx]  # [0,1]
            heat = make_gradcam_heatmap(img, model, last_conv)
            heatmap = (heat * 255).astype(np.uint8)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            base = (np.clip(img * 255, 0, 255)).astype(np.uint8)
            overlay = cv2.addWeighted(base, 0.6, heatmap, 0.4, 0)

            fn = f"{tag}_{i:03d}_true{id2name[int(y_true_int[idx])]}_pred{id2name[int(y_pred_int[idx])]}.png"
            cv2.imwrite(os.path.join(out_dir, tag, fn), overlay[:, :, ::-1])  # save as RGB

    print(f"[Grad-CAM] Saved to {out_dir}")

generate_gradcam_for_images(
    model=cnn,
    X_float01=X_test_np,
    y_true_int=y_test_np,
    y_pred_int=y_pred_test,
    id2name=id2name,
    last_conv=LAST_CONV,
    out_dir="explain_out/gradcam",
    num_each=5
)


In [ ]:
def generate_gradient_input_explanations(model, X_float01, y_true_int, y_pred_int, id2name, num_images=10):
    idxs = np.arange(len(X_float01))[:num_images]
    for i, idx in enumerate(idxs):
        x = tf.convert_to_tensor(X_float01[idx:idx+1], dtype=tf.float32)  # [0,1]
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
            top = tf.argmax(preds[0])
            loss = preds[:, top]
        grads = tape.gradient(loss, x)[0]
        gi = (grads * x[0]).numpy()
        gi -= gi.mean(); gi /= (gi.std() + 1e-8); gi = np.clip(gi * 0.1 + 0.5, 0, 1)
        gi = (gi * 255).astype(np.uint8)

        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.imshow((X_float01[idx]*255).astype(np.uint8)); 
        plt.title(f"True: {id2name[int(y_true_int[idx])]}\nPred: {id2name[int(y_pred_int[idx])]}")
        plt.axis('off')
        plt.subplot(1,2,2); plt.imshow(gi); plt.title("Gradient × Input"); plt.axis('off')
        plt.tight_layout()
        plt.savefig(f"explain_out/gradcam/gradinput_{i:03d}.png", dpi=160)
        plt.show()

generate_gradient_input_explanations(cnn, X_test_np, y_test_np, y_pred_test, id2name, num_images=10)


In [ ]:
os.makedirs("explain_out/misclassified", exist_ok=True)
os.makedirs("explain_out/correct", exist_ok=True)

mis_idx = np.where(y_pred_test != y_test_np)[0]
cor_idx = np.where(y_pred_test == y_test_np)[0]

def save_image_uint8(img_float01, path):
    img = (np.clip(img_float01*255, 0, 255)).astype(np.uint8)
    cv2.imwrite(path, img[:, :, ::-1])  # RGB→BGR for cv2

for i, idx in enumerate(mis_idx[:50]):  # limit to 50 to avoid dumping too many files
    save_image_uint8(X_test_np[idx], f"explain_out/misclassified/img_{i:03d}_true{id2name[int(y_test_np[idx])]}_pred{id2name[int(y_pred_test[idx])]}.png")

for i, idx in enumerate(cor_idx[:50]):
    save_image_uint8(X_test_np[idx], f"explain_out/correct/img_{i:03d}_true{id2name[int(y_test_np[idx])]}_pred{id2name[int(y_pred_test[idx])]}.png")
